## Data Loading and Corridor Station Extraction

This section prepares the input data needed for the $8$-cell / $30$-second $CTM$ benchmark. It loads the selected $PeMS$ detector metadata, filters the full $5$-minute station dataset, and creates the two main time windows used later in the benchmark.

### What the Code Does

The code first defines the final set of selected detector IDs:

- $7$ mainline detectors
- $5$ on-ramp detectors
- $3$ off-ramp detectors

These detectors represent the selected $I$-$405$ southbound corridor.

It then loads the station metadata file and extracts the important metadata fields:

- $station\_id$
- $station\_name$
- $station\_type$
- $freeway$
- $direction$
- $absolute\_postmile$
- $length$
- $lanes$
- $latitude$
- $longitude$

The metadata is filtered so only the selected corridor stations remain. Numeric fields are converted into numeric format, and the stations are sorted by $absolute\_postmile$ so the corridor order is clear.

### Free-Flow Data Window

The code loads the full $PeMS$ $5$-minute station dataset and extracts the early-morning window:

$01{:}00$ to $05{:}00$

This low-congestion window is used later to estimate free-flow speeds. The selected columns are:

- $timestamp$
- $station\_id$
- $station\_type$
- $total\_flow$
- $avg\_occupancy$
- $avg\_speed$

This creates `selected_data_midnight`.

### Morning Benchmark Window

The code also extracts the main benchmark period:

$08{:}00$ to $09{:}00$

This is the peak-hour window used for the final $CTM$ benchmark. Since $PeMS$ data is reported every $5$ minutes, this gives $12$ time intervals:

$08{:}00, 08{:}05, \ldots, 08{:}55$

Each $5$-minute interval will later be expanded into ten $30$-second $CTM$ steps, giving:

$12 \times 10 = 120$ simulation steps

This creates `selected_data_morning`.

### Column Reference

The final table documents which raw $PeMS$ columns are used:

| Raw Column | Meaning |
|---:|---|
| $0$ | $timestamp$ |
| $1$ | $station\_id$ |
| $5$ | $station\_type$ |
| $9$ | $total\_flow$ |
| $10$ | $avg\_occupancy$ |
| $11$ | $avg\_speed$ |

### Output and Interpretation

This section produces two cleaned datasets:

- `selected_data_midnight`: low-congestion data for estimating free-flow speed
- `selected_data_morning`: $08{:}00$–$09{:}00$ benchmark data for the final $CTM$ simulation

These datasets are the foundation for all later calculations, including free-flow speed estimation, mainline state initialization, ramp release series, off-ramp flow series, and the full $120$-step state-based benchmark.

In [40]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os
%matplotlib inline


In [41]:
# 1. Read station metadata and display selected stations clearly
import pandas as pd

# Final selected detector IDs:
# 7 mainline + 5 on-ramp + 3 off-ramp
ids_to_keep = [
    1201419, 1201469, 1201497, 1201525, 1201558, 1201589, 1201620,
    1201460, 1201490, 1201517, 1201548, 1201580,
    1201465, 1201554, 1201585
]

ids_to_keep_str = {str(x).strip() for x in ids_to_keep}




ids_to_keep_str = {str(x).strip() for x in ids_to_keep}

metadata_raw = pd.read_csv(
    "Station Metadata_district12.txt",
    sep=None,
    engine="python",
    header=None,
    dtype=str,
    skip_blank_lines=True
)

metadata_raw = metadata_raw.apply(lambda col: col.astype(str).str.strip())

selected_metadata_clean = pd.DataFrame({
    "station_id": metadata_raw[0],
    "station_name": metadata_raw[13],
    "station_type": metadata_raw[11],
    "freeway": metadata_raw[1],
    "direction": metadata_raw[2],
    "absolute_postmile": metadata_raw[7],
    "length": metadata_raw[10],
    "lanes": metadata_raw[12],
    "latitude": metadata_raw[8],
    "longitude": metadata_raw[9],
})

# Keep only selected stations
selected_metadata_clean["station_id"] = selected_metadata_clean["station_id"].astype(str).str.strip()

selected_metadata_clean = selected_metadata_clean[
    selected_metadata_clean["station_id"].isin(ids_to_keep_str)
].copy()

# Convert numeric columns
numeric_cols = [
    "station_id",
    "freeway",
    "absolute_postmile",
    "length",
    "lanes",
    "latitude",
    "longitude",
]

for col in numeric_cols:
    selected_metadata_clean[col] = pd.to_numeric(selected_metadata_clean[col], errors="coerce")

# Sort by corridor position
selected_metadata_clean = selected_metadata_clean.sort_values(
    ["absolute_postmile", "station_type"]
).reset_index(drop=True)

print("Selected Station Metadata ")
print("Number of selected stations:", len(selected_metadata_clean))

display(selected_metadata_clean)

Selected Station Metadata 
Number of selected stations: 15


,station_id,station_name,station_type,freeway,direction,absolute_postmile,length,lanes,latitude,longitude
0,1201419,RED HILL,ML,405,S,8.17,0.269,5,33.686517,-117.866474
1,1201465,BRISTOL 1,FR,405,S,9.31,NaN,2,33.687251,-117.886180
2,1201469,BRISTOL 1,ML,405,S,9.31,0.350,5,33.687251,-117.886180
3,1201460,BRISTOL 1,OR,405,S,9.31,NaN,1,33.687251,-117.886180
4,1201497,FAIRVIEW,ML,405,S,10.05,0.460,5,33.687480,-117.899035
5,1201490,FAIRVIEW,OR,405,S,10.07,NaN,1,33.687494,-117.899383
6,1201525,HARBOR 1,ML,405,S,10.97,0.610,6,33.687942,-117.915002
7,1201517,HARBOR 1,OR,405,S,10.97,NaN,1,33.687942,-117.915002
8,1201554,HARBOR 2,FR,405,S,11.27,NaN,3,33.689212,-117.919950
9,1201558,HARBOR 2,ML,405,S,11.27,0.480,5,33.689212,-117.919950


In [42]:

# 2. Load PeMS 5-minute station data
# Purpose:
# Load the full 5-minute detector dataset and create the 01:00–05:00
# free-flow window used to estimate free-flow speed.

# Load 5-minute PeMS station data
station_data = pd.read_csv("station_5min_district12.txt", header=None)

# Column 0 = timestamp
station_data[0] = pd.to_datetime(station_data[0])

# Final selected detector IDs:
# 7 mainline + 5 on-ramp + 3 off-ramp
ids_to_keep = [
    1201419, 1201469, 1201497, 1201525, 1201558, 1201589, 1201620,
    1201460, 1201490, 1201517, 1201548, 1201580,
    1201465, 1201554, 1201585
]

# Free-flow window
# This window is used only to estimate free-flow speed from low-congestion conditions.
start_time = "2026-01-08 01:00:00"
end_time   = "2026-01-08 05:00:00"

# Filter selected detectors during free-flow window
filtered_data_midnight = station_data[
    (station_data[1].isin(ids_to_keep)) &
    (station_data[0] >= pd.Timestamp(start_time)) &
    (station_data[0] <  pd.Timestamp(end_time))
].copy()

# Keep only useful columns
selected_data_midnight = filtered_data_midnight[[0, 1, 5, 9, 10, 11]].copy()

selected_data_midnight.columns = [
    "timestamp",
    "station_id",
    "station_type",
    "total_flow",
    "avg_occupancy",
    "avg_speed"
]

# Sort for readability
selected_data_midnight = selected_data_midnight.sort_values(
    ["timestamp", "station_id"]
).reset_index(drop=True)

print("Midnight / Free-Flow Window Data ")
print("Time window: 2026-01-08 01:00:00 to 2026-01-08 05:00:00")
print("Rows:", len(selected_data_midnight))
print("Unique timestamps:", selected_data_midnight["timestamp"].nunique())
print("Unique stations:", selected_data_midnight["station_id"].nunique())

print("\nFirst 20 rows:")
display(selected_data_midnight.head(20))

print("\nRows for station 1201419 (RED HILL):")
display(selected_data_midnight[selected_data_midnight["station_id"] == 1201419].head(20))

Midnight / Free-Flow Window Data 
Time window: 2026-01-08 01:00:00 to 2026-01-08 05:00:00
Rows: 720
Unique timestamps: 48
Unique stations: 15

First 20 rows:


,timestamp,station_id,station_type,total_flow,avg_occupancy,avg_speed
0,2026-01-08 01:00:00,1201419,ML,112.0,0.0345,47.3
1,2026-01-08 01:00:00,1201460,OR,0.0,0.0000,NaN
2,2026-01-08 01:00:00,1201465,FR,NaN,NaN,NaN
3,2026-01-08 01:00:00,1201469,ML,112.0,0.0257,61.4
4,2026-01-08 01:00:00,1201490,OR,9.0,0.0200,NaN
5,2026-01-08 01:00:00,1201497,ML,55.0,0.0049,69.8
6,2026-01-08 01:00:00,1201517,OR,3.0,0.0030,NaN
7,2026-01-08 01:00:00,1201525,ML,54.0,0.0097,64.5
8,2026-01-08 01:00:00,1201548,OR,6.0,0.0110,NaN
9,2026-01-08 01:00:00,1201554,FR,10.0,0.0033,NaN



Rows for station 1201419 (RED HILL):


,timestamp,station_id,station_type,total_flow,avg_occupancy,avg_speed
0,2026-01-08 01:00:00,1201419,ML,112.0,0.0345,47.3
15,2026-01-08 01:05:00,1201419,ML,174.0,0.0332,53.4
30,2026-01-08 01:10:00,1201419,ML,177.0,0.0398,53.7
45,2026-01-08 01:15:00,1201419,ML,217.0,0.0335,64.4
60,2026-01-08 01:20:00,1201419,ML,206.0,0.0344,68.4
75,2026-01-08 01:25:00,1201419,ML,194.0,0.0342,68.2
90,2026-01-08 01:30:00,1201419,ML,179.0,0.0312,68.0
105,2026-01-08 01:35:00,1201419,ML,155.0,0.0361,60.9
120,2026-01-08 01:40:00,1201419,ML,184.0,0.0401,57.7
135,2026-01-08 01:45:00,1201419,ML,188.0,0.0335,63.1


In [43]:
# 3. Load 08:00–09:00 benchmark window data
# Purpose:
# This is the actual peak-hour window used for the 8-cell / 30-sec CTM benchmark.
# PeMS gives 5-minute data, so 08:00–09:00 gives 12 intervals:
# 08:00, 08:05, ..., 08:55
# Each 5-min interval will later be divided into ten 30-sec CTM steps.

start_time = "2026-01-08 08:00:00"
end_time   = "2026-01-08 09:00:00"

filtered_data_morning = station_data[
    (station_data[1].isin(ids_to_keep)) &
    (station_data[0] >= pd.Timestamp(start_time)) &
    (station_data[0] <  pd.Timestamp(end_time))
].copy()

selected_data_morning = filtered_data_morning[[0, 1, 5, 9, 10, 11]].copy()

selected_data_morning.columns = [
    "timestamp",
    "station_id",
    "station_type",
    "total_flow",
    "avg_occupancy",
    "avg_speed"
]

selected_data_morning = selected_data_morning.sort_values(
    ["timestamp", "station_id"]
).reset_index(drop=True)

print(" Morning Benchmark Window Data ")
print("Time window: 2026-01-08 08:00:00 to 2026-01-08 09:00:00")
print("Rows:", len(selected_data_morning))
print("Unique timestamps:", selected_data_morning["timestamp"].nunique())
print("Unique stations:", selected_data_morning["station_id"].nunique())

print("\nFirst 30 rows:")
display(selected_data_morning.head(30))

print("\nRows for station 1201419 (RED HILL)")
display(selected_data_morning[selected_data_morning["station_id"] == 1201419])

 Morning Benchmark Window Data 
Time window: 2026-01-08 08:00:00 to 2026-01-08 09:00:00
Rows: 180
Unique timestamps: 12
Unique stations: 15

First 30 rows:


,timestamp,station_id,station_type,total_flow,avg_occupancy,avg_speed
0,2026-01-08 08:00:00,1201419,ML,645.0,0.1582,39.4
1,2026-01-08 08:00:00,1201460,OR,88.0,0.0840,NaN
2,2026-01-08 08:00:00,1201465,FR,NaN,NaN,NaN
3,2026-01-08 08:00:00,1201469,ML,681.0,0.1384,40.0
4,2026-01-08 08:00:00,1201490,OR,39.0,0.8700,NaN
5,2026-01-08 08:00:00,1201497,ML,622.0,0.0525,69.5
6,2026-01-08 08:00:00,1201517,OR,73.0,0.0980,NaN
7,2026-01-08 08:00:00,1201525,ML,737.0,0.1677,30.6
8,2026-01-08 08:00:00,1201548,OR,56.0,0.0980,NaN
9,2026-01-08 08:00:00,1201554,FR,82.0,0.0283,NaN



Rows for station 1201419 (RED HILL)


,timestamp,station_id,station_type,total_flow,avg_occupancy,avg_speed
0,2026-01-08 08:00:00,1201419,ML,645.0,0.1582,39.4
15,2026-01-08 08:05:00,1201419,ML,677.0,0.1536,39.8
30,2026-01-08 08:10:00,1201419,ML,703.0,0.1708,39.6
45,2026-01-08 08:15:00,1201419,ML,769.0,0.1629,43.6
60,2026-01-08 08:20:00,1201419,ML,726.0,0.1420,44.4
75,2026-01-08 08:25:00,1201419,ML,757.0,0.1616,47.5
90,2026-01-08 08:30:00,1201419,ML,772.0,0.1622,46.8
105,2026-01-08 08:35:00,1201419,ML,670.0,0.1642,44.0
120,2026-01-08 08:40:00,1201419,ML,715.0,0.1535,43.7
135,2026-01-08 08:45:00,1201419,ML,587.0,0.1551,38.0


In [44]:
# 5. Column reference for PeMS station_5min data

pems_column_reference = pd.DataFrame({
    "column_index": [0, 1, 5, 9, 10, 11],
    "column_name": [
        "timestamp",
        "station_id",
        "station_type",
        "total_flow",
        "avg_occupancy",
        "avg_speed"
    ]

})

display(pems_column_reference)

,column_index,column_name
0,0,timestamp
1,1,station_id
2,5,station_type
3,9,total_flow
4,10,avg_occupancy
5,11,avg_speed


## Free-Flow Speed, Travel Time, and Flow-Based Delay Sanity Check

This section estimates free-flow conditions and computes a flow/speed-based mainline delay check for the selected $I$-$405$ corridor. This is used as a sanity check before the full state-based $CTM$ benchmark.

### Free-Flow Speed Estimation

The code uses the low-congestion window from $01{:}00$ to $05{:}00$ to estimate free-flow speed at each mainline detector.

For each mainline station, the median speed is computed:

$ \tilde{v}_s = \text{median}(avg\_speed_s) $

where:

- $ \tilde{v}_s $ = free-flow speed estimate at station $s$
- $avg\_speed_s$ = observed speed samples during the low-congestion window

The median is used because it is more stable against noisy detector readings.

### Segment Free-Flow Speed

Each freeway segment lies between two neighboring mainline stations. The segment free-flow speed is estimated using the harmonic mean:

$ v_i^{ff} = \frac{2v_{up}v_{down}}{v_{up}+v_{down}} $

where:

- $v_i^{ff}$ = free-flow speed of segment $i$
- $v_{up}$ = upstream station free-flow speed
- $v_{down}$ = downstream station free-flow speed

The harmonic mean is appropriate because speed is a rate.

### Free-Flow Travel Time

For each segment, free-flow travel time is computed as:

$TT_i^{ff} = \frac{L_i}{v_i^{ff}}$

where:

- $TT_i^{ff}$ = free-flow travel time for segment $i$
- $L_i$ = segment length in miles
- $v_i^{ff}$ = segment free-flow speed in mph

The code stores travel time in both hours and minutes.

### Observed Travel Time at $08{:}00$

The code then extracts observed mainline speeds at exactly $08{:}00$ and computes observed segment speed using the same harmonic mean:

$ v_i^{obs} = \frac{2v_{up}^{obs}v_{down}^{obs}}{v_{up}^{obs}+v_{down}^{obs}} $

Observed travel time is then:

$TT_i^{obs} = \frac{L_i}{v_i^{obs}}$

where:

- $TT_i^{obs}$ = observed travel time for segment $i$
- $v_i^{obs}$ = observed segment speed at $08{:}00$

### Segment Delay

Per-vehicle delay is computed as:

$d_i = TT_i^{obs} - TT_i^{ff}$

where:

- $d_i$ = delay per vehicle on segment $i$
- $TT_i^{obs}$ = observed travel time
- $TT_i^{ff}$ = free-flow travel time

The code calculates delay in both minutes and hours.

### Flow-Based Mainline Delay

The code uses downstream mainline detector flow as the segment outflow:

$Q_{out,i}$

Then total flow-based delay is:

$D_{M,i} = d_i \cdot Q_{out,i}$

where:

- $D_{M,i}$ = total mainline delay for segment $i$
- $d_i$ = delay per vehicle
- $Q_{out,i}$ = downstream mainline flow during the $5$-minute interval

The total corridor delay is:

$D_M = \sum_i D_{M,i}$

### Output and Interpretation

This section produces:

- free-flow speed by mainline station
- segment free-flow speed
- segment free-flow travel time
- observed $08{:}00$ travel time
- per-vehicle segment delay
- total flow-based mainline delay

This result is only a flow/speed-based sanity check using one $08{:}00$ snapshot. It is not the final benchmark objective. The official benchmark is the later state-based $CTM$ simulation over $120$ steps from $08{:}00$ to $09{:}00$.

In [45]:

# 5. Estimate free-flow speed for each mainline station
# Purpose:
# Use the low-congestion midnight window, 01:00–05:00,
# to estimate free-flow speed at each mainline detector.

mainline_ids = [
    1201419, 1201469, 1201497, 1201525, 1201558, 1201589, 1201620
]

mainline_midnight_data = selected_data_midnight[
    (selected_data_midnight["station_id"].isin(mainline_ids)) &
    (selected_data_midnight["station_type"] == "ML")
].copy()

mainline_midnight_data["avg_speed"] = pd.to_numeric(
    mainline_midnight_data["avg_speed"],
    errors="coerce"
)

median_speed_by_station = (
    mainline_midnight_data
    .groupby("station_id", as_index=False)["avg_speed"]
    .median()
)

# IMPORTANT:
# Keep this column name as "median_speed"
# because the next block uses median_speed_df["median_speed"]
median_speed_by_station.columns = ["station_id", "median_speed"]

# Optional clean display table, but do NOT change median_speed_by_station
median_speed_display = selected_metadata_clean[
    selected_metadata_clean["station_id"].isin(mainline_ids)
][
    ["station_id", "station_name", "absolute_postmile"]
].merge(
    median_speed_by_station,
    on="station_id",
    how="left"
).sort_values("absolute_postmile").reset_index(drop=True)

print(" Median Free-Flow Speed by Mainline Station ")
display(median_speed_display)

 Median Free-Flow Speed by Mainline Station 


,station_id,station_name,absolute_postmile,median_speed
0,1201419,RED HILL,8.17,64.20
1,1201469,BRISTOL 1,9.31,65.35
2,1201497,FAIRVIEW,10.05,69.85
3,1201525,HARBOR 1,10.97,68.40
4,1201558,HARBOR 2,11.27,68.70
5,1201589,EUCLID,12.27,68.05
6,1201620,TALBERT,13.07,67.95


In [46]:

# Verify median free-flow speed calculation
mainline_midnight_check = selected_data_midnight[
    (selected_data_midnight["station_id"].isin(mainline_ids)) &
    (selected_data_midnight["station_type"] == "ML")
].copy()

mainline_midnight_check["avg_speed"] = pd.to_numeric(
    mainline_midnight_check["avg_speed"],
    errors="coerce"
)

speed_check = (
    mainline_midnight_check
    .groupby("station_id")["avg_speed"]
    .agg(
        count="count",
        min_speed="min",
        median_speed="median",
        max_speed="max"
    )
    .reset_index()
)

speed_check = speed_check.merge(
    selected_metadata_clean[["station_id", "station_name"]],
    on="station_id",
    how="left"
)

speed_check = speed_check[
    ["station_id", "station_name", "count", "min_speed", "median_speed", "max_speed"]
]

display(speed_check)

,station_id,station_name,count,min_speed,median_speed,max_speed
0,1201419,RED HILL,48,47.3,64.20,78.7
1,1201469,BRISTOL 1,48,49.0,65.35,69.4
2,1201497,FAIRVIEW,48,39.7,69.85,74.4
3,1201525,HARBOR 1,48,63.4,68.40,72.9
4,1201558,HARBOR 2,48,64.9,68.70,73.8
5,1201589,EUCLID,48,63.4,68.05,74.3
6,1201620,TALBERT,48,62.5,67.95,73.9


In [47]:
# 6. Segment free-flow speed calculation
# Purpose:
# Use the median free-flow speed at each pair of neighboring
# mainline stations to estimate segment free-flow speed.

# segment free flow speed
def segment_free_flow_speed(upstream_id, downstream_id, median_speed_df):
    v_upstream = median_speed_df.loc[
        median_speed_df["station_id"] == upstream_id,
        "median_speed"
    ].iloc[0]

    v_downstream = median_speed_df.loc[
        median_speed_df["station_id"] == downstream_id,
        "median_speed"
    ].iloc[0]

    return (2 * v_upstream * v_downstream) / (v_upstream + v_downstream)

# free flow speed between each segment
v_ff_1 = segment_free_flow_speed(1201419, 1201469, median_speed_by_station)
v_ff_2 = segment_free_flow_speed(1201469, 1201497, median_speed_by_station)
v_ff_3 = segment_free_flow_speed(1201497, 1201525, median_speed_by_station)
v_ff_4 = segment_free_flow_speed(1201525, 1201558, median_speed_by_station)
v_ff_5 = segment_free_flow_speed(1201558, 1201589, median_speed_by_station)
v_ff_6 = segment_free_flow_speed(1201589, 1201620, median_speed_by_station)

segment_free_flow_df = pd.DataFrame({
    "segment": ["S1", "S2", "S3", "S4", "S5", "S6"],
    "from_station": ["RED HILL", "BRISTOL 1", "FAIRVIEW", "HARBOR 1", "HARBOR 2", "EUCLID"],
    "to_station": ["BRISTOL 1", "FAIRVIEW", "HARBOR 1", "HARBOR 2", "EUCLID", "TALBERT"],
    "v_ff_mph": [v_ff_1, v_ff_2, v_ff_3, v_ff_4, v_ff_5, v_ff_6]
})
segment_free_flow_df["v_ff_mph"] = segment_free_flow_df["v_ff_mph"].round(3)
display(segment_free_flow_df)

,segment,from_station,to_station,v_ff_mph
0,S1,RED HILL,BRISTOL 1,64.770
1,S2,BRISTOL 1,FAIRVIEW,67.525
2,S3,FAIRVIEW,HARBOR 1,69.117
3,S4,HARBOR 1,HARBOR 2,68.550
4,S5,HARBOR 2,EUCLID,68.373
5,S6,EUCLID,TALBERT,68.000


In [48]:
# 7. Free-flow travel time calculation
# Purpose:
# Convert each segment's free-flow speed into free-flow travel time.

# Formula:
# travel_time_hours = segment_length_miles / free_flow_speed_mph
# travel_time_minutes = travel_time_hours * 60
# segment length in miles

L_1 = 1.14
L_2 = 0.74
L_3 = 0.92
L_4 = 0.30
L_5 = 1.00
L_6 = 0.80

def segment_free_flow_travel_time(length, segment_free_flow_speed):
    travel_time_hour = length / segment_free_flow_speed
    travel_time_min = travel_time_hour * 60
    return travel_time_hour, travel_time_min

TT_ff_1_hr, TT_ff_1_min = segment_free_flow_travel_time(L_1, v_ff_1)
TT_ff_2_hr, TT_ff_2_min = segment_free_flow_travel_time(L_2, v_ff_2)
TT_ff_3_hr, TT_ff_3_min = segment_free_flow_travel_time(L_3, v_ff_3)
TT_ff_4_hr, TT_ff_4_min = segment_free_flow_travel_time(L_4, v_ff_4)
TT_ff_5_hr, TT_ff_5_min = segment_free_flow_travel_time(L_5, v_ff_5)
TT_ff_6_hr, TT_ff_6_min = segment_free_flow_travel_time(L_6, v_ff_6)

total_h = TT_ff_1_hr + TT_ff_2_hr + TT_ff_3_hr + TT_ff_4_hr + TT_ff_5_hr + TT_ff_6_hr
total_min = TT_ff_1_min + TT_ff_2_min + TT_ff_3_min + TT_ff_4_min + TT_ff_5_min + TT_ff_6_min

free_flow_tt_df = pd.DataFrame({
    "segment": [
        "Segment 1: RED HILL → BRISTOL 1",
        "Segment 2: BRISTOL 1 → FAIRVIEW",
        "Segment 3: FAIRVIEW → HARBOR 1",
        "Segment 4: HARBOR 1 → HARBOR 2",
        "Segment 5: HARBOR 2 → EUCLID",
        "Segment 6: EUCLID → TALBERT",
    ],
    "length_miles": [L_1, L_2, L_3, L_4, L_5, L_6],
    "free_flow_speed_mph": [v_ff_1, v_ff_2, v_ff_3, v_ff_4, v_ff_5, v_ff_6],
    "TT_ff_hr": [TT_ff_1_hr, TT_ff_2_hr, TT_ff_3_hr, TT_ff_4_hr, TT_ff_5_hr, TT_ff_6_hr],
    "TT_ff_min": [TT_ff_1_min, TT_ff_2_min, TT_ff_3_min, TT_ff_4_min, TT_ff_5_min, TT_ff_6_min],
})

print("Segment Free-Flow Travel Time ")
display(free_flow_tt_df)

print("Total free-flow travel time:")
print("Hours:", round(total_h, 3))
print("Minutes:", round(total_min, 3))

Segment Free-Flow Travel Time 


,segment,length_miles,free_flow_speed_mph,TT_ff_hr,TT_ff_min
0,Segment 1: RED HILL → BRISTOL 1,1.14,64.769896,0.017601,1.056046
1,Segment 2: BRISTOL 1 → FAIRVIEW,0.74,67.525111,0.010959,0.657533
2,Segment 3: FAIRVIEW → HARBOR 1,0.92,69.117396,0.013311,0.798641
3,Segment 4: HARBOR 1 → HARBOR 2,0.30,68.549672,0.004376,0.262583
4,Segment 5: HARBOR 2 → EUCLID,1.00,68.373455,0.014626,0.877534
5,Segment 6: EUCLID → TALBERT,0.80,67.999963,0.011765,0.705883


Total free-flow travel time:
Hours: 0.073
Minutes: 4.358


### Segment Speed Calculation Note

For the flow-based sanity check, segment speed is estimated using the harmonic mean of the upstream and downstream detector speeds:

$$[
v_{\text{seg}} = \frac{2v_{\text{up}}v_{\text{down}}}{v_{\text{up}} + v_{\text{down}}}
]$$

The harmonic mean is used because speeds are rates. This is more appropriate than a simple arithmetic mean when estimating travel-time-related quantities.

In [49]:
# Flow/speed-based sanity check: observed travel time at 08:00

mainline_ids = [
    1201419,
    1201469,
    1201497,
    1201525,
    1201558,
    1201589,
    1201620
]


# Get mainline station speeds at exactly 08:00
mainline_ids_8am = selected_data_morning[
    (selected_data_morning["timestamp"] == pd.Timestamp("2026-01-08 08:00:00")) &
    (selected_data_morning["station_id"].isin(mainline_ids)) &
    (selected_data_morning["station_type"] == "ML")
].copy()


mainline_ids_8am["avg_speed"] = pd.to_numeric(
    mainline_ids_8am["avg_speed"],
    errors="coerce"
)


def get_station_speed(station_id, df):
    station_speed = df.loc[
        df["station_id"] == station_id,
        "avg_speed"
    ]

    if station_speed.empty:
        raise ValueError(f"Missing speed for station {station_id}")

    return station_speed.iloc[0]


def observed_segment_speed(upstream_id, downstream_id, df):
    v_upstream = get_station_speed(upstream_id, df)
    v_downstream = get_station_speed(downstream_id, df)

    return (2 * v_upstream * v_downstream) / (v_upstream + v_downstream)


# Observed segment speeds at 08:00
v_obs_1 = observed_segment_speed(1201419, 1201469, mainline_ids_8am)
v_obs_2 = observed_segment_speed(1201469, 1201497, mainline_ids_8am)
v_obs_3 = observed_segment_speed(1201497, 1201525, mainline_ids_8am)
v_obs_4 = observed_segment_speed(1201525, 1201558, mainline_ids_8am)
v_obs_5 = observed_segment_speed(1201558, 1201589, mainline_ids_8am)
v_obs_6 = observed_segment_speed(1201589, 1201620, mainline_ids_8am)


def segment_observed_travel_time(length, observed_segment_speed):
    travel_time_hour = length / observed_segment_speed
    travel_time_min = travel_time_hour * 60

    return travel_time_hour, travel_time_min


# Observed travel time by segment
TT_obs_1_hr, TT_obs_1_min = segment_observed_travel_time(L_1, v_obs_1)
TT_obs_2_hr, TT_obs_2_min = segment_observed_travel_time(L_2, v_obs_2)
TT_obs_3_hr, TT_obs_3_min = segment_observed_travel_time(L_3, v_obs_3)
TT_obs_4_hr, TT_obs_4_min = segment_observed_travel_time(L_4, v_obs_4)
TT_obs_5_hr, TT_obs_5_min = segment_observed_travel_time(L_5, v_obs_5)
TT_obs_6_hr, TT_obs_6_min = segment_observed_travel_time(L_6, v_obs_6)


observed_travel_time_sanity_check_df = pd.DataFrame({
    "segment": [
        "Segment 1: RED HILL → BRISTOL 1",
        "Segment 2: BRISTOL 1 → FAIRVIEW",
        "Segment 3: FAIRVIEW → HARBOR 1",
        "Segment 4: HARBOR 1 → HARBOR 2",
        "Segment 5: HARBOR 2 → EUCLID",
        "Segment 6: EUCLID → TALBERT",
    ],
    "observed_speed_mph": [
        v_obs_1,
        v_obs_2,
        v_obs_3,
        v_obs_4,
        v_obs_5,
        v_obs_6,
    ],
    "observed_travel_time_min": [
        TT_obs_1_min,
        TT_obs_2_min,
        TT_obs_3_min,
        TT_obs_4_min,
        TT_obs_5_min,
        TT_obs_6_min,
    ],
    "observed_travel_time_hr": [
        TT_obs_1_hr,
        TT_obs_2_hr,
        TT_obs_3_hr,
        TT_obs_4_hr,
        TT_obs_5_hr,
        TT_obs_6_hr,
    ],
})

print(" Flow/Speed-Based Observed Travel Time Sanity Check ")
display(observed_travel_time_sanity_check_df.round(3))

 Flow/Speed-Based Observed Travel Time Sanity Check 


,segment,observed_speed_mph,observed_travel_time_min,observed_travel_time_hr
0,Segment 1: RED HILL → BRISTOL 1,39.698,1.723,0.029
1,Segment 2: BRISTOL 1 → FAIRVIEW,50.776,0.874,0.015
2,Segment 3: FAIRVIEW → HARBOR 1,42.492,1.299,0.022
3,Segment 4: HARBOR 1 → HARBOR 2,29.078,0.619,0.010
4,Segment 5: HARBOR 2 → EUCLID,24.708,2.428,0.040
5,Segment 6: EUCLID → TALBERT,26.840,1.788,0.030


# flow-based travel-time benchmark
This calculation is included only as a PeMS speed-based sanity check.
It estimates per-vehicle corridor delay using observed PeMS speeds and free-flow travel time.
It is not used as the final benchmark objective because it does not include CTM states, ramp queues, fairness penalties, or capacity/spillback penalties.

In [50]:

# Flow/speed-based sanity check: segment delay
# Purpose:
# This calculates per-vehicle delay using:
#     delay = observed travel time - free-flow travel time

def segment_delay(segment_observed_travel_time, segment_free_flow_travel_time):
    segment_delay = segment_observed_travel_time - segment_free_flow_travel_time
    return segment_delay


# Segment delay in hours
delay_1 = segment_delay(TT_obs_1_hr, TT_ff_1_hr)
delay_2 = segment_delay(TT_obs_2_hr, TT_ff_2_hr)
delay_3 = segment_delay(TT_obs_3_hr, TT_ff_3_hr)
delay_4 = segment_delay(TT_obs_4_hr, TT_ff_4_hr)
delay_5 = segment_delay(TT_obs_5_hr, TT_ff_5_hr)
delay_6 = segment_delay(TT_obs_6_hr, TT_ff_6_hr)



# Segment delay in minutes
delay_1_min = segment_delay(TT_obs_1_min, TT_ff_1_min)
delay_2_min = segment_delay(TT_obs_2_min, TT_ff_2_min)
delay_3_min = segment_delay(TT_obs_3_min, TT_ff_3_min)
delay_4_min = segment_delay(TT_obs_4_min, TT_ff_4_min)
delay_5_min = segment_delay(TT_obs_5_min, TT_ff_5_min)
delay_6_min = segment_delay(TT_obs_6_min, TT_ff_6_min)


# Total corridor delay

total_delay_hours = (
    delay_1
    + delay_2
    + delay_3
    + delay_4
    + delay_5
    + delay_6
)

total_delay_min = (
    delay_1_min
    + delay_2_min
    + delay_3_min
    + delay_4_min
    + delay_5_min
    + delay_6_min
)


# Clean display table
segment_delay_sanity_check_df = pd.DataFrame({
    "segment": [
        "Segment 1: RED HILL → BRISTOL 1",
        "Segment 2: BRISTOL 1 → FAIRVIEW",
        "Segment 3: FAIRVIEW → HARBOR 1",
        "Segment 4: HARBOR 1 → HARBOR 2",
        "Segment 5: HARBOR 2 → EUCLID",
        "Segment 6: EUCLID → TALBERT",
    ],
    "observed_travel_time_min": [
        TT_obs_1_min,
        TT_obs_2_min,
        TT_obs_3_min,
        TT_obs_4_min,
        TT_obs_5_min,
        TT_obs_6_min,
    ],
    "free_flow_travel_time_min": [
        TT_ff_1_min,
        TT_ff_2_min,
        TT_ff_3_min,
        TT_ff_4_min,
        TT_ff_5_min,
        TT_ff_6_min,
    ],
    "delay_min": [
        delay_1_min,
        delay_2_min,
        delay_3_min,
        delay_4_min,
        delay_5_min,
        delay_6_min,
    ],
    "delay_hr": [
        delay_1,
        delay_2,
        delay_3,
        delay_4,
        delay_5,
        delay_6,
    ],
})

print(" Flow/Speed-Based Segment Delay Sanity Check ")
display(segment_delay_sanity_check_df.round(3))

print("Total delay hours:", round(total_delay_hours, 3))
print("Total delay minutes:", round(total_delay_min, 3))

 Flow/Speed-Based Segment Delay Sanity Check 


,segment,observed_travel_time_min,free_flow_travel_time_min,delay_min,delay_hr
0,Segment 1: RED HILL → BRISTOL 1,1.723,1.056,0.667,0.011
1,Segment 2: BRISTOL 1 → FAIRVIEW,0.874,0.658,0.217,0.004
2,Segment 3: FAIRVIEW → HARBOR 1,1.299,0.799,0.500,0.008
3,Segment 4: HARBOR 1 → HARBOR 2,0.619,0.263,0.356,0.006
4,Segment 5: HARBOR 2 → EUCLID,2.428,0.878,1.551,0.026
5,Segment 6: EUCLID → TALBERT,1.788,0.706,1.083,0.018


Total delay hours: 0.073
Total delay minutes: 4.374


In [51]:
# Total mainline delay for each segment
# delay_i is in vehicle-hours per vehicle
# flow_i is vehicles during the 5-minute interval
# result is vehicle-hours during the 5-minute interval
def total_mainline_delay(segment_delay, total_flow):
    total_mainline_delay = segment_delay * total_flow
    return total_mainline_delay


In [52]:
# total mainline delay
# Formula:  total_delay_i = delay_i * Q_out_i

def total_mainline_delay(segment_delay, total_flow):
    total_mainline_delay = segment_delay * total_flow
    return total_mainline_delay



# Q_out_i = downstream mainline station flow for segment i at 08:00
mainline_flow_8am = selected_data_morning[
    (selected_data_morning["timestamp"] == pd.Timestamp("2026-01-08 08:00:00")) &
    (selected_data_morning["station_type"] == "ML")
].copy()

mainline_flow_8am["total_flow"] = pd.to_numeric(
    mainline_flow_8am["total_flow"],
    errors="coerce"
)


def get_station_flow(station_id, df):
    station_flow = df.loc[
        df["station_id"] == station_id,
        "total_flow"
    ]

    if station_flow.empty:
        raise ValueError(f"Missing flow for station {station_id}")

    return station_flow.iloc[0]


# Downstream station flow for each segment
Q_out_1 = get_station_flow(1201469, mainline_flow_8am)  # BRISTOL 1
Q_out_2 = get_station_flow(1201497, mainline_flow_8am)  # FAIRVIEW
Q_out_3 = get_station_flow(1201525, mainline_flow_8am)  # HARBOR 1
Q_out_4 = get_station_flow(1201558, mainline_flow_8am)  # HARBOR 2
Q_out_5 = get_station_flow(1201589, mainline_flow_8am)  # EUCLID
Q_out_6 = get_station_flow(1201620, mainline_flow_8am)  # TALBERT


# Mainline delay in vehicle-minutes
mainline_delay_1_min = total_mainline_delay(delay_1_min, Q_out_1)
mainline_delay_2_min = total_mainline_delay(delay_2_min, Q_out_2)
mainline_delay_3_min = total_mainline_delay(delay_3_min, Q_out_3)
mainline_delay_4_min = total_mainline_delay(delay_4_min, Q_out_4)
mainline_delay_5_min = total_mainline_delay(delay_5_min, Q_out_5)
mainline_delay_6_min = total_mainline_delay(delay_6_min, Q_out_6)


# Mainline delay in vehicle-hours
mainline_delay_1_hr = mainline_delay_1_min / 60
mainline_delay_2_hr = mainline_delay_2_min / 60
mainline_delay_3_hr = mainline_delay_3_min / 60
mainline_delay_4_hr = mainline_delay_4_min / 60
mainline_delay_5_hr = mainline_delay_5_min / 60
mainline_delay_6_hr = mainline_delay_6_min / 60


# Total flow-based mainline delay
total_flow_based_delay_min = sum([
    mainline_delay_1_min,
    mainline_delay_2_min,
    mainline_delay_3_min,
    mainline_delay_4_min,
    mainline_delay_5_min,
    mainline_delay_6_min,
])

total_flow_based_delay_hr = total_flow_based_delay_min / 60


# Clean display table
flow_based_mainline_delay_df = pd.DataFrame({
    "segment": [
        "Segment 1: RED HILL → BRISTOL 1",
        "Segment 2: BRISTOL 1 → FAIRVIEW",
        "Segment 3: FAIRVIEW → HARBOR 1",
        "Segment 4: HARBOR 1 → HARBOR 2",
        "Segment 5: HARBOR 2 → EUCLID",
        "Segment 6: EUCLID → TALBERT",
    ],
    "Q_out_vehicles_5min": [
        Q_out_1,
        Q_out_2,
        Q_out_3,
        Q_out_4,
        Q_out_5,
        Q_out_6,
    ],
    "delay_min_per_vehicle": [
        delay_1_min,
        delay_2_min,
        delay_3_min,
        delay_4_min,
        delay_5_min,
        delay_6_min,
    ],
    "mainline_delay_vehicle_min": [
        mainline_delay_1_min,
        mainline_delay_2_min,
        mainline_delay_3_min,
        mainline_delay_4_min,
        mainline_delay_5_min,
        mainline_delay_6_min,
    ],
    "mainline_delay_vehicle_hr": [
        mainline_delay_1_hr,
        mainline_delay_2_hr,
        mainline_delay_3_hr,
        mainline_delay_4_hr,
        mainline_delay_5_hr,
        mainline_delay_6_hr,
    ],
})

print(" Flow/Speed-Based Mainline Delay Sanity Check ")
display(flow_based_mainline_delay_df.round(3))

print("Total flow-based delay in vehicle-min:", round(total_flow_based_delay_min, 3))
print("Total flow-based delay in vehicle-hr:", round(total_flow_based_delay_hr, 3))

 Flow/Speed-Based Mainline Delay Sanity Check 


,segment,Q_out_vehicles_5min,delay_min_per_vehicle,mainline_delay_vehicle_min,mainline_delay_vehicle_hr
0,Segment 1: RED HILL → BRISTOL 1,681.0,0.667,454.209,7.570
1,Segment 2: BRISTOL 1 → FAIRVIEW,622.0,0.217,134.906,2.248
2,Segment 3: FAIRVIEW → HARBOR 1,737.0,0.500,368.826,6.147
3,Segment 4: HARBOR 1 → HARBOR 2,599.0,0.356,213.510,3.559
4,Segment 5: HARBOR 2 → EUCLID,680.0,1.551,1054.537,17.576
5,Segment 6: EUCLID → TALBERT,550.0,1.083,595.384,9.923


Total flow-based delay in vehicle-min: 2821.373
Total flow-based delay in vehicle-hr: 47.023


## State-Based $CTM$ Benchmark Setup and Simulation

This section builds the official $8$-cell, $30$-second state-based $CTM$ benchmark for the $08{:}00$–$09{:}00$ morning window. The benchmark uses observed $PeMS$ data as the field-observed ramp-release baseline and computes mainline delay, ramp delay, fairness penalty, capacity penalties, and the final raw objective.

---

## $08{:}00$–$09{:}00$ Benchmark Input Series

The code first extracts the full morning benchmark window:

$08{:}00 \leq t < 09{:}00$

Since $PeMS$ reports data every $5$ minutes, the one-hour window gives:

$12$ intervals

Each $5$-minute interval is divided into ten $30$-second steps:

$12 \times 10 = 120$ $CTM$ steps

The helper function $build\_30sec\_series\_from\_5min\_flow$ converts each $5$-minute flow value into a repeated $30$-second flow series:

$q^{30sec} = \frac{q^{5min}}{10}$

The code builds:

- $q\_{in,boundary}$ for Cell $1$
- observed ramp release series $u_1$ to $u_5$
- ramp arrival series $a_j(t)$
- off-ramp flow series $f\_{out}$

Ramp arrival is assumed as:

$a_j(t) = 1.5 \cdot u_j^{obs}(t)$

This same arrival assumption is used consistently across the benchmark, $ADMM$, and $ADMM$-$MPC$ models.

---

## Ramp Queue Capacity and Fairness Setup

Ramp maximum queue storage is estimated from ramp length, lane count, and average vehicle length:

$R_{max} = \frac{\text{ramp length} \times \text{number of lanes}}{\text{average vehicle length}}$

where the average vehicle length is:

$25$ ft

The code creates two queue-capacity mappings:

- $ramp\_max\_queue\_named$ for real ramp names
- $ramp\_max\_queue\_by\_u$ for control variables $u_1$ to $u_5$

The fairness penalty uses ramp stress:

$\phi_i = \frac{R_i}{R_{max,i}}$

Stress is capped at $1$:

$\phi_i^{cap} = \min(\phi_i, 1)$

The fairness penalty is:

$L_{fair} = \gamma \sum_{i<j}(\phi_i^{cap} - \phi_j^{cap})^2$

This penalizes uneven ramp queue burden across ramps.

---

## Ramp Queue Update and Local Ramp Delay

Ramp queues are updated each $30$-second step using:

$R_t = R_{t-1} + a_t - u_t$

The queue is capped by physical ramp storage:

$R_t = \min(R_t, R_{max})$

If the uncapped queue exceeds $R_{max}$, the excess becomes spillback.

Local ramp delay is computed using the average queue during the step:

$D_L(t) = \left(\frac{R_{t-1} + R_t}{2}\right)\Delta t$

where:

$\Delta t = 0.5$ minutes

---

## Doorway, Physical, and Safe Capacity Setup

Doorway capacity is estimated from known station capacities. Known $5$-minute doorway capacities are converted into per-lane values, averaged, and then converted to $30$-second $CTM$ units.

The final doorway capacity for each cell is:

$C_i = C_{\text{per-lane},30sec} \times n_i$

Physical capacity is computed using jam density:

$N_{max,i} = k_j \cdot L_i \cdot n_i$

where:

- $k_j = 193$ veh/mi/lane
- $L_i$ = cell length
- $n_i$ = number of lanes

Safe-threshold capacity is:

$X_{safe,i} = \eta N_{max,i}$

with:

$\eta = 0.7$

The lane map uses Cell $5$ and Cell $7$ as $6$-lane cells, consistent with the corrected metadata.

---

## Capacity Penalty Function

The capacity penalty combines four terms.

Doorway penalty:

$L_{doorway,i} = \lambda_1 \max(q_{in,i} + u_i - C_i, 0)^2$

Safe-threshold penalty:

$L_{safe,i} = \lambda_2 \max(x_i - X_{safe,i}, 0)^2$

Physical-capacity penalty:

$L_{physical,i} = \lambda_3 \max(x_i - N_{max,i}, 0)^2$

Spillback penalty:

$L_{spillback,j} = \lambda_4 \cdot spillback_j^2$

The total capacity penalty is:

$L_{capacity} = L_{doorway} + L_{safe} + L_{physical} + L_{spillback}$

---

## $CTM$ State Update

The $CTM$ update uses sending and receiving logic.

Sending flow:

$S_i = \min(x_i, C_i)$

Receiving flow:

$R_i = \max(0, \min(C_i, N_{max,i} - x_i))$

Mainline outflow from Cell $i$ is limited by upstream sending and downstream receiving:

$q_{out,i} = \min(S_i, R_{i+1})$

The state update is:

$x_{i,t+1} = x_{i,t} + q_{in,i} + u_i - q_{out,i} - f_i$

The code clips negative states to zero for numerical safety.

---

## Mainline Delay

Each cell’s state-based mainline delay is computed as:

$D_i = TTT_i - FF_i$

where:

$TTT_i = \left(\frac{x_{i,t} + x_{i,t+1}}{2}\right)\Delta t$

and:

$FF_i = (q_{out,i} + f_i)TT_i^{ff}$

Because physical delay cannot be negative, the final delay is clamped:

$D_i = \max(TTT_i - FF_i, 0)$

This avoids artificial negative delay from draining cells.

---

## Full $120$-Step Benchmark Simulation

The function $simulate\_state\_based\_benchmark\_120\_steps$ runs the full field-observed ramp-release benchmark over all $120$ steps.

At each step, it:

1. reads dynamic $PeMS$ inputs
2. updates ramp queues and spillback
3. computes local ramp delay
4. computes fairness penalty
5. updates the mainline $CTM$ state
6. computes mainline delay
7. computes capacity penalties
8. stores all results in $history$

The raw objective per step is:

$J_t = D_M(t) + D_L(t) + L_{fair}(t) + L_{capacity}(t)$

The final raw objective is:

$J = \sum_t J_t$

---

## Output and Interpretation

The final summary reports:

- mainline delay
- local ramp delay
- fairness penalty
- doorway penalty
- safe-threshold penalty
- physical-capacity penalty
- spillback penalty
- total capacity penalty
- raw total objective

It also reports the final mainline state $x$ and final ramp queues $R$ after $120$ simulation steps.

This benchmark represents the official field-observed ramp-release state-based $CTM$ baseline. It is the comparison point for the later $ADMM$ and $ADMM$-$MPC$ ramp-control models.

In [53]:
#1. build 08:00–09:00 benchmark window
start_time = "2026-01-08 08:00:00"
end_time   = "2026-01-08 09:00:00"

filtered_data_morning = station_data[
    (station_data[1].isin(ids_to_keep)) &
    (station_data[0] >= pd.Timestamp(start_time)) &
    (station_data[0] <  pd.Timestamp(end_time))
].copy()

selected_data_morning = filtered_data_morning[[0, 1, 5, 9, 10, 11]].copy()

selected_data_morning.columns = [
    "timestamp",
    "station_id",
    "station_type",
    "total_flow",
    "avg_occupancy",
    "avg_speed"
]

selected_data_morning["total_flow"] = pd.to_numeric(
    selected_data_morning["total_flow"],
    errors="coerce"
)

selected_data_morning = selected_data_morning.sort_values(
    ["timestamp", "station_id"]
).reset_index(drop=True)

print("Morning Benchmark Window")
print("Rows:", len(selected_data_morning))
print("Unique timestamps:", selected_data_morning["timestamp"].nunique())
print("Unique stations:", selected_data_morning["station_id"].nunique())

display(
    selected_data_morning
    .groupby("station_id")
    .size()
    .reset_index(name="row_count")
)

Morning Benchmark Window
Rows: 180
Unique timestamps: 12
Unique stations: 15


,station_id,row_count
0,1201419,12
1,1201460,12
2,1201465,12
3,1201469,12
4,1201490,12
5,1201497,12
6,1201517,12
7,1201525,12
8,1201548,12
9,1201554,12


# Build 120-Step CTM Input Series from 08:00–09:00 PeMS Data

## Purpose

This section converts PeMS 5-minute flow data into 30-second CTM input series for the official 8-cell, 30-second, 120-step state-based benchmark.

PeMS reports one flow value every 5 minutes. Since the CTM simulation uses a 30-second timestep, each 5-minute PeMS flow is converted into ten 30-second values:

$$
q_{30s} = \frac{q_{5min}}{10}
$$

Each converted 30-second value is then repeated for ten CTM steps.

Because the benchmark covers one hour:

$$
60 \text{ minutes} \div 0.5 \text{ minutes per step} = 120 \text{ CTM steps}
$$

## Input Series Created

This section creates the following 120-step input series:

- `q_in_boundary_series`: boundary inflow entering Cell 1
- `observed_release_series`: observed on-ramp releases for ramps `u1`–`u5`
- `ramp_arrival_series`: assumed ramp arrivals, defined as `1.5 × observed_release_series`
- `f_out_series`: off-ramp flows mapped to CTM cells

## Ramp Arrival Assumption

PeMS provides observed ramp release flow, but it does not directly observe true ramp arrival demand. Therefore, ramp arrivals are estimated as:

$$
a_j(t) = 1.5 \cdot u^{obs}_j(t)
$$

where:

- $a_j(t)$ = assumed arrival demand at ramp $j$ during time step $t$
- $u^{obs}_j(t)$ = observed ramp release at ramp $j$ during time step $t$

This same `arrival_multiplier = 1.5` assumption is used consistently across the benchmark, ADMM, and ADMM-MPC experiments.

## Off-Ramp Mapping

Off-ramp flows are mapped into CTM cells as follows:

- Bristol 1 FR → Cell 2
- Harbor 2 FR → Cell 6
- Euclid FR → Cell 7

Missing off-ramp flow values are filled with `0.0`.

## Verification

At the end of this section, the notebook checks that each generated input series has exactly 120 values and displays the first 30-second input values used at the beginning of the simulation.

In [54]:
# 2. Build 120-step CTM input series from 08:00–09:00 PeMS data
num_steps = 120
delta_t = 0.5  # 30 seconds = 0.5 minutes


# Get one PeMS station's 5-minute flow  and turn it into a 120-step 30-second CTM.
def build_30sec_series_from_5min_flow(station_id, selected_data_morning):
    station_5min_data = selected_data_morning[
        selected_data_morning["station_id"] == station_id
    ].copy()

    station_5min_data = station_5min_data.sort_values(
        "timestamp"
    ).reset_index(drop=True)

    station_5min_data["total_flow"] = pd.to_numeric(
        station_5min_data["total_flow"],
        errors="coerce"
    ).fillna(0.0)

    flow_30sec_series = []

    for flow_5min in station_5min_data["total_flow"]:
        flow_30sec = flow_5min / 10.0

        for _ in range(10):
            flow_30sec_series.append(flow_30sec)

    if len(flow_30sec_series) != num_steps:
        raise ValueError(
            f"Station {station_id} produced {len(flow_30sec_series)} steps, "
            f"but expected {num_steps}."
        )

    return flow_30sec_series


# Boundary inflow into Cell 1 , Station 1201419
q_in_boundary_series = build_30sec_series_from_5min_flow(
    1201419,
    selected_data_morning
)


# Observed on-ramp releases
# These are the field-observed ramp-release benchmark ramp releases.
observed_release_series = {
    "u1": build_30sec_series_from_5min_flow(1201460, selected_data_morning),  # BRISTOL 1 OR
    "u2": build_30sec_series_from_5min_flow(1201490, selected_data_morning),  # FAIRVIEW OR
    "u3": build_30sec_series_from_5min_flow(1201517, selected_data_morning),  # HARBOR 1 OR
    "u4": build_30sec_series_from_5min_flow(1201548, selected_data_morning),  # HARBOR 2 OR
    "u5": build_30sec_series_from_5min_flow(1201580, selected_data_morning),  # EUCLID OR
}


# Ramp arrivals
# arrival demand is arrival_multiplier times observed release.
# we made same assumption across all the calculations

arrival_multiplier = 1.5
ramp_arrival_series = {}
for ramp in observed_release_series:
    ramp_arrival_series[ramp] = [
        arrival_multiplier * value
        for value in observed_release_series[ramp]
    ]


# Off-ramp flows mapped into CTM cells
# Missing BRISTOL off-ramp values are filled as 0.

f_out_bristol_series = build_30sec_series_from_5min_flow(
    1201465,
    selected_data_morning
)

f_out_harbor2_series = build_30sec_series_from_5min_flow(
    1201554,
    selected_data_morning
)

f_out_euclid_series = build_30sec_series_from_5min_flow(
    1201585,
    selected_data_morning
)

f_out_series = []

for step in range(num_steps):
    f_out_series.append({
        "Cell 1": 0.0,
        "Cell 2": f_out_bristol_series[step],
        "Cell 3": 0.0,
        "Cell 4": 0.0,
        "Cell 5": 0.0,
        "Cell 6": f_out_harbor2_series[step],
        "Cell 7": f_out_euclid_series[step],
        "Cell 8": 0.0,
    })



In [55]:
# Clean input-series check
input_series_length_check_df = pd.DataFrame({
    "series_name": [
        "q_in_boundary_series",
        "f_out_series",
        "u1 observed_release",
        "u2 observed_release",
        "u3 observed_release",
        "u4 observed_release",
        "u5 observed_release",
    ],
    "length": [
        len(q_in_boundary_series),
        len(f_out_series),
        len(observed_release_series["u1"]),
        len(observed_release_series["u2"]),
        len(observed_release_series["u3"]),
        len(observed_release_series["u4"]),
        len(observed_release_series["u5"]),
    ],
})

first_step_input_df = pd.DataFrame({
    "input_type": [
        "boundary inflow",
        "observed release",
        "observed release",
        "observed release",
        "observed release",
        "observed release",
        "ramp arrival",
        "ramp arrival",
        "ramp arrival",
        "ramp arrival",
        "ramp arrival",
        "off-ramp flow",
        "off-ramp flow",
        "off-ramp flow",
        "off-ramp flow",
        "off-ramp flow",
        "off-ramp flow",
        "off-ramp flow",
        "off-ramp flow",
    ],
    "location": [
        "Cell 1 boundary",
        "u1",
        "u2",
        "u3",
        "u4",
        "u5",
        "u1",
        "u2",
        "u3",
        "u4",
        "u5",
        "Cell 1",
        "Cell 2",
        "Cell 3",
        "Cell 4",
        "Cell 5",
        "Cell 6",
        "Cell 7",
        "Cell 8",
    ],
    "first_30sec_value": [
        q_in_boundary_series[0],
        observed_release_series["u1"][0],
        observed_release_series["u2"][0],
        observed_release_series["u3"][0],
        observed_release_series["u4"][0],
        observed_release_series["u5"][0],
        ramp_arrival_series["u1"][0],
        ramp_arrival_series["u2"][0],
        ramp_arrival_series["u3"][0],
        ramp_arrival_series["u4"][0],
        ramp_arrival_series["u5"][0],
        f_out_series[0]["Cell 1"],
        f_out_series[0]["Cell 2"],
        f_out_series[0]["Cell 3"],
        f_out_series[0]["Cell 4"],
        f_out_series[0]["Cell 5"],
        f_out_series[0]["Cell 6"],
        f_out_series[0]["Cell 7"],
        f_out_series[0]["Cell 8"],
    ],
})

print("120-Step CTM Input Series Length Check")
display(input_series_length_check_df)
print(" First 30-Second CTM Input Values ")
display(first_step_input_df.round(3))

120-Step CTM Input Series Length Check


,series_name,length
0,q_in_boundary_series,120
1,f_out_series,120
2,u1 observed_release,120
3,u2 observed_release,120
4,u3 observed_release,120
5,u4 observed_release,120
6,u5 observed_release,120


 First 30-Second CTM Input Values 


,input_type,location,first_30sec_value
0,boundary inflow,Cell 1 boundary,64.50
1,observed release,u1,8.80
2,observed release,u2,3.90
3,observed release,u3,7.30
4,observed release,u4,5.60
5,observed release,u5,6.40
6,ramp arrival,u1,13.20
7,ramp arrival,u2,5.85
8,ramp arrival,u3,10.95
9,ramp arrival,u4,8.40


## Ramp Maximum Queue Capacity

## Purpose

This block estimates the maximum number of vehicles that each on-ramp can physically store before spillback occurs.

The maximum ramp queue capacity is calculated as:

$$
R_{max,i} = \frac{L_i \times n_i}{l_{veh}}
$$

where:

- $R_{max,i}$ = maximum queue storage for ramp $i$, measured in vehicles
- $L_i$ = ramp length, measured in feet
- $n_i$ = number of ramp lanes
- $l_{veh}$ = assumed average vehicle length, measured in feet per vehicle

In this model, the average vehicle length is assumed to be:

$$
l_{veh} = 25 \text{ ft/vehicle}
$$

## Ramp Storage Inputs

The ramp storage calculation uses:

- ramp length in feet
- number of ramp lanes
- assumed average vehicle length

For example, Euclid has 2 ramp lanes, so its storage capacity is larger than it would be with only one ramp lane.

## Output Dictionaries

The calculated ramp storage capacities are stored in three formats:

1. `ramp_max_queue_named`

   This dictionary uses real ramp names such as `"Bristol 1"` and `"Fairview"`.

   It is used by the fairness penalty calculation because fairness is reported by real ramp name.

2. `ramp_max_queue_by_u`

   This dictionary uses control variable names such as `"u1"` through `"u5"`.

   It is used by the ramp queue update and spillback calculations.

3. `ramp_name_map`

   This dictionary maps each control variable name to its real ramp name.

   For example:

   ```python
   "u1" -> "Bristol 1"

In [56]:
# 3. Ramp maximum queue capacity for fairness penalty
# Purpose: Estimate maximum ramp queue storage using:
ave_veh_length = 25  # feet per vehicle


ramp_length = {
    "Bristol 1": 716.73,
    "Fairview": 1808.89,
    "Harbor 1": 1404.20,
    "Harbor 2": 1811.02,
    "Euclid": 610.24,
}


ramp_lane_count  = {
    "Bristol 1": 1,
    "Fairview": 1,
    "Harbor 1": 1,
    "Harbor 2": 1,
    "Euclid": 2,
}


def R_max(ave_veh_length, ramp_length, ramp_lane_count ):
    R_max = {}

    for ramps in ramp_length:
        R_max[ramps] = (
            ramp_length[ramps] * ramp_lane_count [ramps]
        ) / ave_veh_length

    return R_max


R_max_value = R_max(
    ave_veh_length,
    ramp_length,
    ramp_lane_count
)


# Named ramp max queue Used by fairness penalty
ramp_max_queue_named = {
    "Bristol 1": R_max_value["Bristol 1"],
    "Fairview": R_max_value["Fairview"],
    "Harbor 1": R_max_value["Harbor 1"],
    "Harbor 2": R_max_value["Harbor 2"],
    "Euclid": R_max_value["Euclid"],
}



# Ramp max queue using u1–u5 names
# Used by ramp queue / spillback calculation
ramp_max_queue_by_u = {
    "u1": R_max_value["Bristol 1"],
    "u2": R_max_value["Fairview"],
    "u3": R_max_value["Harbor 1"],
    "u4": R_max_value["Harbor 2"],
    "u5": R_max_value["Euclid"],
}


# Map u1–u5 control names to real ramp names
# Used by fairness_penalty_one_step(...)
ramp_name_map = {
    "u1": "Bristol 1",
    "u2": "Fairview",
    "u3": "Harbor 1",
    "u4": "Harbor 2",
    "u5": "Euclid",
}

# Clean display

ramp_capacity_df = pd.DataFrame({
    "ramp": list(ramp_max_queue_named.keys()),
    "ramp_length_ft": [
        ramp_length["Bristol 1"],
        ramp_length["Fairview"],
        ramp_length["Harbor 1"],
        ramp_length["Harbor 2"],
        ramp_length["Euclid"],
    ],
    "lanes": [
        ramp_lane_count ["Bristol 1"],
        ramp_lane_count ["Fairview"],
        ramp_lane_count ["Harbor 1"],
        ramp_lane_count ["Harbor 2"],
        ramp_lane_count ["Euclid"],
    ],
    "max_queue_vehicles": [
        ramp_max_queue_named["Bristol 1"],
        ramp_max_queue_named["Fairview"],
        ramp_max_queue_named["Harbor 1"],
        ramp_max_queue_named["Harbor 2"],
        ramp_max_queue_named["Euclid"],
    ],
})

print(" Ramp Maximum Queue Capacity ")
display(ramp_capacity_df.round(3))

 Ramp Maximum Queue Capacity 


,ramp,ramp_length_ft,lanes,max_queue_vehicles
0,Bristol 1,716.73,1,28.669
1,Fairview,1808.89,1,72.356
2,Harbor 1,1404.20,1,56.168
3,Harbor 2,1811.02,1,72.441
4,Euclid,610.24,2,48.819


## Fairness Penalty and Ramp Queue Spillback Functions

## Purpose

This section defines the ramp queue update, spillback, and fairness penalty functions used in the official state-based CTM benchmark.

The benchmark uses the same fairness penalty structure as the ADMM and ADMM-MPC models.

## Fairness Penalty

For each ramp, a capped queue stress index is calculated as:

$$
\phi_i(t) = \min\left(\frac{R_i(t)}{R_{max,i}}, 1\right)
$$

where:

- $R_i(t)$ = queue length at ramp $i$ during time step $t$
- $R_{max,i}$ = maximum storage capacity of ramp $i$
- $\phi_i(t)$ = capped ramp stress index

The stress value is capped at 1 so that any ramp queue at or above its physical storage capacity is treated as full saturation.

The fairness penalty is calculated as:

$$
L_{fair}(t) = \gamma \sum_{i<j} \left(\phi_i(t) - \phi_j(t)\right)^2
$$

where:

- $\gamma$ = fairness penalty weight
- $\phi_i(t)$ and $\phi_j(t)$ = capped stress values for two different ramps

This penalty increases when ramp queue stress is uneven across ramps. If one ramp is much closer to saturation than another ramp, the fairness penalty becomes larger.

In this benchmark:

$$
\gamma = 1
$$

## Ramp Queue Update with Spillback

Each ramp queue is updated using the current queue, assumed ramp arrival demand, and observed ramp release:

$$
R^{raw}_i(t+1) = R_i(t) + a_i(t) - u_i(t)
$$

where:

- $R_i(t)$ = current ramp queue
- $a_i(t)$ = assumed ramp arrival demand
- $u_i(t)$ = observed ramp release
- $R^{raw}_i(t+1)$ = raw next queue before applying non-negativity and storage limits

The queue cannot be negative, so the uncapped queue is:

$$
R^{uncapped}_i(t+1) = \max\left(R^{raw}_i(t+1), 0\right)
$$

If the uncapped queue exceeds the maximum ramp storage capacity, the excess vehicles are counted as spillback:

$$
spillback_i(t) = \max\left(R^{uncapped}_i(t+1) - R_{max,i}, 0\right)
$$

The stored ramp queue is then capped at the physical ramp storage limit:

$$
R_i(t+1) = \min\left(R^{uncapped}_i(t+1), R_{max,i}\right)
$$

## First-Step Verification

The first-step check verifies the ramp queue update and fairness penalty using the first 30-second CTM step.

At the first step, all ramp queues start at zero. Since the assumed ramp arrival is larger than the observed release, each ramp queue increases slightly. No spillback occurs during this first step because all ramp queues remain below their maximum storage capacities.

The first-step fairness result is:

$$
fairness\_sum = 0.049
$$

Since $\gamma = 1$, the weighted fairness penalty is:

$$
L_{fair} = 0.049
$$

This confirms that the fairness function is working correctly: the first-step penalty is small because all ramps have low stress values and none are near full saturation.


In [57]:
# 4. Fairness penalty setup for CTM benchmark
#fairness multiplayer (same value used for all the set ups)
gamma = 1

def fairness_penalty_one_step(
    R_next,
    ramp_name_map,
    ramp_max_queue_named,
    gamma
):

    # 1. Compute capped stress index for each ramp
    stress_dict = {}

    for u_name, R_t in R_next.items():
        ramp_name = ramp_name_map[u_name]
        R_max_i = ramp_max_queue_named[ramp_name]

        raw_stress = R_t / R_max_i
        capped_stress = min(raw_stress, 1.0)

        stress_dict[ramp_name] = capped_stress


    # 2. Compute pairwise fairness penalty
    ramps = list(stress_dict.keys())
    fairness_sum = 0.0

    for i in range(len(ramps)):
        for j in range(i + 1, len(ramps)):
            phi_i = stress_dict[ramps[i]]
            phi_j = stress_dict[ramps[j]]

            fairness_sum += (phi_i - phi_j) ** 2


    # 3. Apply fairness weight
    L_fair = gamma * fairness_sum
    return stress_dict, fairness_sum, L_fair

In [58]:
# 5. Conservative ramp queue update with persistent spillback/external backlog
def ramp_next_queue_with_spillback(
    R_current,
    B_current,
    ramp_arrival_step,
    observed_release_step,
    ramp_max_queue_by_u
):
    R_next = {}
    B_next = {}
    spillback_by_ramp = {}
    actual_release_step = {}

    for ramp in R_current:

        R_now = float(R_current[ramp])
        B_now = float(B_current.get(ramp, 0.0))
        arrival = float(ramp_arrival_step[ramp])
        commanded_release = float(observed_release_step[ramp])
        R_max = float(ramp_max_queue_by_u[ramp])

        # All waiting demand available before this step's release.
        # This includes vehicles already stored on the ramp, vehicles that
        # spilled back previously, and this step's new arrivals.
        available = max(0.0, R_now) + max(0.0, B_now) + max(0.0, arrival)

        # The ramp cannot release more vehicles than actually exist.
        actual_release = min(
            max(0.0, commanded_release),
            available
        )

        waiting_after_release = max(
            0.0,
            available - actual_release
        )

        # Store as many waiting vehicles as the physical ramp can hold.
        R_next[ramp] = min(
            waiting_after_release,
            R_max
        )

        # Excess vehicles remain in an external spillback/backlog queue.
        B_next[ramp] = max(
            0.0,
            waiting_after_release - R_max
        )

        # For compatibility with the existing capacity penalty, spillback is
        # the persistent external queue state at the end of the timestep.
        spillback_by_ramp[ramp] = B_next[ramp]
        actual_release_step[ramp] = actual_release

    return R_next, B_next, spillback_by_ramp, actual_release_step


In [59]:
# 6. Fairness penalty setup check
# Use first 30-second benchmark step
fairness_test_step = 0

# If ramp_queue_0 is not already defined, start queues at zero
if "ramp_queue_0" not in globals():
    ramp_queue_0 = {
        "u1": 0.0,
        "u2": 0.0,
        "u3": 0.0,
        "u4": 0.0,
        "u5": 0.0,
    }

R_current_test = ramp_queue_0.copy()
B_current_test = {
    ramp: 0.0
    for ramp in R_current_test
}

ramp_arrival_step_test = {
    "u1": ramp_arrival_series["u1"][fairness_test_step],
    "u2": ramp_arrival_series["u2"][fairness_test_step],
    "u3": ramp_arrival_series["u3"][fairness_test_step],
    "u4": ramp_arrival_series["u4"][fairness_test_step],
    "u5": ramp_arrival_series["u5"][fairness_test_step],
}

observed_release_step_test = {
    "u1": observed_release_series["u1"][fairness_test_step],
    "u2": observed_release_series["u2"][fairness_test_step],
    "u3": observed_release_series["u3"][fairness_test_step],
    "u4": observed_release_series["u4"][fairness_test_step],
    "u5": observed_release_series["u5"][fairness_test_step],
}

# next ramp queue for first step
(
    R_next_test,
    B_next_test,
    spillback_by_ramp_test,
    actual_release_step_test,
) = ramp_next_queue_with_spillback(
    R_current_test,
    B_current_test,
    ramp_arrival_step_test,
    observed_release_step_test,
    ramp_max_queue_by_u
)

# fairness penalty for first step
stress_dict_test, fairness_sum_test, L_fair_test = fairness_penalty_one_step(
    R_next_test,
    ramp_name_map,
    ramp_max_queue_named,
    gamma
)

# result
fairness_check_rows = []
for u_name in R_next_test:
    ramp_name = ramp_name_map[u_name]
    R_max_i = ramp_max_queue_named[ramp_name]

    raw_stress = R_next_test[u_name] / R_max_i
    capped_stress = min(raw_stress, 1.0)

    fairness_check_rows.append({
        "u_name": u_name,
        "ramp_name": ramp_name,
        "R_prev": R_current_test[u_name],
        "B_prev": B_current_test[u_name],
        "arrival": ramp_arrival_step_test[u_name],
        "commanded_release": observed_release_step_test[u_name],
        "actual_release": actual_release_step_test[u_name],
        "R_next": R_next_test[u_name],
        "B_next": B_next_test[u_name],
        "total_waiting_next": R_next_test[u_name] + B_next_test[u_name],
        "R_max": R_max_i,
        "raw_stress": raw_stress,
        "capped_stress": capped_stress,
        "spillback": spillback_by_ramp_test[u_name],
    })

fairness_check_df = pd.DataFrame(fairness_check_rows)
print(" Fairness Penalty First-Step Check")
display(fairness_check_df.round(3))

print("gamma:", gamma)
print("fairness_sum:", round(fairness_sum_test, 3))
print("L_fair:", round(L_fair_test, 3))


 Fairness Penalty First-Step Check


,u_name,ramp_name,R_prev,B_prev,arrival,commanded_release,actual_release,R_next,B_next,total_waiting_next,R_max,raw_stress,capped_stress,spillback
0,u1,Bristol 1,0.0,0.0,13.20,8.8,8.8,4.40,0.0,4.40,28.669,0.153,0.153,0.0
1,u2,Fairview,0.0,0.0,5.85,3.9,3.9,1.95,0.0,1.95,72.356,0.027,0.027,0.0
2,u3,Harbor 1,0.0,0.0,10.95,7.3,7.3,3.65,0.0,3.65,56.168,0.065,0.065,0.0
3,u4,Harbor 2,0.0,0.0,8.40,5.6,5.6,2.80,0.0,2.80,72.441,0.039,0.039,0.0
4,u5,Euclid,0.0,0.0,9.60,6.4,6.4,3.20,0.0,3.20,48.819,0.066,0.066,0.0


gamma: 1
fairness_sum: 0.049
L_fair: 0.049


### Doorway Capacity Calculation

## Purpose

The doorway capacity represents the maximum number of vehicles that can pass through each mainline CTM cell during one 30-second timestep.

In the PeMS data, doorway capacities are available at selected detector stations in units of vehicles per 5-minute interval. Since the final CTM benchmark uses a 30-second timestep, these station-level 5-minute capacities are converted into cell-level 30-second capacities.

## Station-Level Per-Lane Capacity

First, the per-lane doorway capacity is calculated for stations where doorway capacity is available:

$$
C_{\text{per-lane},s}
=
\frac{C_s}{N_s}
$$

where:

- $C_s$ = doorway capacity at station $s$, measured in vehicles per 5 minutes
- $N_s$ = number of mainline lanes at station $s$
- $C_{\text{per-lane},s}$ = per-lane doorway capacity at station $s$, measured in vehicles per 5 minutes

## Estimating Missing Station Capacities

RED HILL and FAIRVIEW do not have given doorway capacities. Their capacities are estimated using the average per-lane doorway capacity from stations with known capacity values:

$$
\bar{C}_{\text{per-lane}}
=
\frac{1}{n}
\sum_{s=1}^{n}
C_{\text{per-lane},s}
$$

The missing station capacity is then estimated as:

$$
C_s
=
\bar{C}_{\text{per-lane}}
\cdot N_s
$$

where:

- $\bar{C}_{\text{per-lane}}$ = average per-lane doorway capacity from known stations
- $N_s$ = number of lanes at the station with missing doorway capacity

## Conversion from 5-Minute Capacity to 30-Second Capacity

The average per-lane capacity is converted from vehicles per 5 minutes to vehicles per 30 seconds:

$$
C_{\text{per-lane},30s}
=
\frac{\bar{C}_{\text{per-lane}}}{5}
\cdot \Delta t
$$

where:

- $\Delta t = 0.5$ minutes
- $C_{\text{per-lane},30s}$ = average per-lane doorway capacity in vehicles per 30 seconds

Since the timestep is 30 seconds, this is equivalent to dividing the 5-minute per-lane capacity by 10.

## Cell-Level Doorway Capacity

Finally, the doorway capacity for each CTM cell is computed using the cell lane count:

$$
C_i
=
C_{\text{per-lane},30s}
\cdot N_i
$$

where:

- $C_i$ = doorway capacity of CTM Cell $i$, measured in vehicles per 30 seconds
- $N_i$ = number of lanes in CTM Cell $i$

The corrected 8-cell lane map is:

```python
cell_lane_count = {
    1: 5,
    2: 5,
    3: 5,
    4: 5,
    5: 6,
    6: 5,
    7: 6,
    8: 5,
}

In [60]:
#7 . Doorway capacity setup for 8-cell / 30-sec CTM
# Purpose:Estimate mainline doorway capacity and convert it into the

doorway_capacity_given = {
    "Bristol 1": 895,
    "Harbor 1": 1029,
    "Harbor 2": 860,
    "Euclid": 925,
    "Talbert": 833
}

# Mainline lane count at detector stations
station_lane_count = {
    "RED HILL": 5,
    "Bristol 1": 5,
    "FAIRVIEW": 5,
    "Harbor 1": 6,
    "Harbor 2": 5,
    "Euclid": 6,
    "Talbert": 5
}


In [61]:

# Compute per-lane doorway capacity
def per_lane_doorway_cap(doorway_capacity_dict, lane_dict):
    per_lane = {}

    for station in doorway_capacity_dict:
        per_lane[station] = (
            doorway_capacity_dict[station] / lane_dict[station]
        )

    return per_lane


def average_per_lane_capacity(per_lane_dict):
    total = 0.0

    for station in per_lane_dict:
        total += per_lane_dict[station]

    return total / len(per_lane_dict)


def doorway_capacity_not_given(avg_per_lane_cap, lane_dict, missing_stations):
    estimated = {}

    for station in missing_stations:
        estimated[station] = avg_per_lane_cap * lane_dict[station]

    return estimated


per_lane_capacity = per_lane_doorway_cap(
    doorway_capacity_given,
    station_lane_count
)

avg_per_lane_cap = average_per_lane_capacity(per_lane_capacity)

missing_stations = ["RED HILL", "FAIRVIEW"]

estimated_capacity = doorway_capacity_not_given(
    avg_per_lane_cap,
    station_lane_count,
    missing_stations
)


In [62]:
# Combine known and estimated station capacities
doorway_capacity_station_5min = {}

for station in doorway_capacity_given:
    doorway_capacity_station_5min[station] = doorway_capacity_given[station]

for station in estimated_capacity:
    doorway_capacity_station_5min[station] = estimated_capacity[station]


In [63]:
# Convert average per-lane capacity to CTM 30-sec units
delta_t = 0.5  # 30 seconds = 0.5 minutes

doorway_capacity_per_lane_per_min = avg_per_lane_cap / 5.0
doorway_capacity_per_lane_30sec = doorway_capacity_per_lane_per_min * delta_t

# 8-cell lane count

cell_lane_count = {
    1: 5,
    2: 5,
    3: 5,
    4: 5,
    5: 6,
    6: 5,
    7: 6,
    8: 5,
}


In [64]:
# Final CTM doorway capacity
doorway_capacity = {
    f"Cell {i}": doorway_capacity_per_lane_30sec * cell_lane_count[i]
    for i in range(1, 9)
}

In [65]:
# results

doorway_station_capacity_df = pd.DataFrame({
    "station": list(doorway_capacity_station_5min.keys()),
    "lanes": [
        station_lane_count[station]
        for station in doorway_capacity_station_5min
    ],
    "doorway_capacity_veh_5min": [
        doorway_capacity_station_5min[station]
        for station in doorway_capacity_station_5min
    ],
    "per_lane_capacity_veh_5min": [
        doorway_capacity_station_5min[station] / station_lane_count[station]
        for station in doorway_capacity_station_5min
    ],
})

doorway_cell_capacity_df = pd.DataFrame({
    "cell": list(doorway_capacity.keys()),
    "lanes": [
        cell_lane_count[i]
        for i in range(1, 9)
    ],
    "doorway_capacity_veh_30sec": [
        doorway_capacity[f"Cell {i}"]
        for i in range(1, 9)
    ],
})

print("Doorway Capacity from PeMS Stations ")
display(doorway_station_capacity_df.round(3))

print("Average per-lane doorway capacity, veh/5-min:", round(avg_per_lane_cap, 3))
print("Average per-lane doorway capacity, veh/min:", round(doorway_capacity_per_lane_per_min, 3))
print("Average per-lane doorway capacity, veh/30-sec:", round(doorway_capacity_per_lane_30sec, 3))

print("\n Final CTM Doorway Capacity by Cell ")
display(doorway_cell_capacity_df.round(3))

Doorway Capacity from PeMS Stations 


,station,lanes,doorway_capacity_veh_5min,per_lane_capacity_veh_5min
0,Bristol 1,5,895.000,179.000
1,Harbor 1,6,1029.000,171.500
2,Harbor 2,5,860.000,172.000
3,Euclid,6,925.000,154.167
4,Talbert,5,833.000,166.600
5,RED HILL,5,843.267,168.653
6,FAIRVIEW,5,843.267,168.653


Average per-lane doorway capacity, veh/5-min: 168.653
Average per-lane doorway capacity, veh/min: 33.731
Average per-lane doorway capacity, veh/30-sec: 16.865

 Final CTM Doorway Capacity by Cell 


,cell,lanes,doorway_capacity_veh_30sec
0,Cell 1,5,84.327
1,Cell 2,5,84.327
2,Cell 3,5,84.327
3,Cell 4,5,84.327
4,Cell 5,6,101.192
5,Cell 6,5,84.327
6,Cell 7,6,101.192
7,Cell 8,5,84.327


### Physical Capacity Setup for the 8-Cell CTM

## Purpose

The old prototype calculated physical capacity for the original 6 detector-to-detector freeway segments. That version is no longer used for the official benchmark because the final CTM model uses 8 equal-length cells with a 30-second timestep.

For the official state-based CTM benchmark, physical capacity is calculated for each CTM cell using:

$$
N^{\max}_i = k_{\text{jam}} \cdot L_i \cdot n_i
$$

where:

- $N^{\max}_i$ = physical capacity of Cell $i$, measured in vehicles
- $k_{\text{jam}}$ = jam density, measured in vehicles per mile per lane
- $L_i$ = CTM cell length, measured in miles
- $n_i$ = number of lanes in Cell $i$

## Corridor Length and Cell Length

The modeled corridor runs from postmile 8.17 to postmile 13.07:

$$
13.07 - 8.17 = 4.90 \text{ miles}
$$

Since the final CTM uses 8 equal-length cells, each cell has length:

$$
L_i = \frac{4.90}{8} = 0.6125 \text{ miles}
$$

## Jam Density Assumption

The jam density is assumed to be:

$$
k_{\text{jam}} = 193 \text{ veh/mi/lane}
$$

## Physical Capacity by Lane Count

For a 5-lane cell, the physical capacity is:

$$
193 \cdot 0.6125 \cdot 5 = 591.0625 \text{ vehicles}
$$

For a 6-lane cell, the physical capacity is:

$$
193 \cdot 0.6125 \cdot 6 = 709.275 \text{ vehicles}
$$

## Corrected Cell Lane Map

The final 8-cell CTM uses the following lane-count map:

```python
station_lane_count = {
    "Cell 1": 5,
    "Cell 2": 5,
    "Cell 3": 5,
    "Cell 4": 5,
    "Cell 5": 6,
    "Cell 6": 5,
    "Cell 7": 6,
    "Cell 8": 5,
}

In [66]:
# 8. Physical capacity setup for 8-cell CTM
# Purpose: Compute the maximum number of vehicles each CTM cell can store.

# from Highway Traffic Manual
jam_density = 193  # veh/mi/lane

# 8-cell corridor
segment_start_PM = 8.17
segment_end_PM = 13.07
num_cells = 8

cell_length = (segment_end_PM - segment_start_PM) / num_cells

# Number of lanes
cell_lane_count_by_cell  = {
    "Cell 1": 5,
    "Cell 2": 5,
    "Cell 3": 5,
    "Cell 4": 5,
    "Cell 5": 6,
    "Cell 6": 5,
    "Cell 7": 6,
    "Cell 8": 5,
}


In [67]:
# Physical capacity calculation
def calculate_physical_capacity(jam_density, cell_length, cell_lane_count_by_cell ):
    N_max_value = {}

    for cell in cell_lane_count_by_cell:
        N_max_value[cell] = (
            jam_density
            * cell_length
            * cell_lane_count_by_cell[cell]
        )

    return N_max_value


N_max_value = calculate_physical_capacity(
    jam_density,
    cell_length,
    cell_lane_count_by_cell
)


# Final CTM physical capacity dictionary
physical_capacity = N_max_value.copy()


In [68]:
# Result
physical_capacity_df = pd.DataFrame({
    "cell": list(physical_capacity.keys()),
    "cell_length_miles": [cell_length for _ in physical_capacity],
    "lanes": [
        cell_lane_count_by_cell[cell]
        for cell in physical_capacity
    ],
    "jam_density_veh_mi_lane": [
        jam_density
        for _ in physical_capacity
    ],
    "physical_capacity_vehicles": [
        physical_capacity[cell]
        for cell in physical_capacity
    ],
})

print("Physical Capacity by 8-Cell CTM Cell ")
print("cell_length:", round(cell_length, 4), "miles")
display(physical_capacity_df.round(3))

Physical Capacity by 8-Cell CTM Cell 
cell_length: 0.6125 miles


,cell,cell_length_miles,lanes,jam_density_veh_mi_lane,physical_capacity_vehicles
0,Cell 1,0.612,5,193,591.062
1,Cell 2,0.612,5,193,591.062
2,Cell 3,0.612,5,193,591.062
3,Cell 4,0.612,5,193,591.062
4,Cell 5,0.612,6,193,709.275
5,Cell 6,0.612,5,193,591.062
6,Cell 7,0.612,6,193,709.275
7,Cell 8,0.612,5,193,591.062


### Safe Occupancy Threshold for 8-Cell CTM

## Purpose

The safe occupancy threshold defines the maximum desired vehicle storage level for each CTM cell before the model begins applying a safety-related capacity penalty.

This threshold is not the same as physical capacity. Physical capacity represents the maximum number of vehicles a cell can store, while the safe threshold represents a lower desired operating limit.

For each cell, the safe threshold is calculated as:

$$
X_{\text{safe}, i} = \eta \cdot N^{\max}_i
$$

where:

- $X_{\text{safe}, i}$ = safe occupancy threshold for Cell $i$, measured in vehicles
- $\eta$ = safe occupancy multiplier
- $N^{\max}_i$ = physical capacity of Cell $i$, measured in vehicles

In this project, the official benchmark uses:

$$
\eta = 0.7
$$

Therefore, the safe threshold is:

$$
X_{\text{safe}, i} = 0.7 \cdot N^{\max}_i
$$

## Why This Threshold Is Used

The same safe threshold value is used across the benchmark, ADMM, and ADMM-MPC models. This keeps the comparison consistent across all experiments.

All models use the same:

- 8-cell CTM structure
- 30-second timestep
- physical capacity values
- safe occupancy threshold
- capacity penalty definition

## Use in the Benchmark

The `safe_threshold_capacity` dictionary stores the safe threshold for each CTM cell.

During the benchmark simulation, if the next cell state exceeds this safe threshold, the model applies a safe-threshold penalty:

$$
P_{\text{safe}, i}(t)
=
\lambda_2
\left[
\max\left(x_i(t+1) - X_{\text{safe}, i}, 0\right)
\right]^2
$$

where:

- $x_i(t+1)$ = next vehicle storage level in Cell $i$
- $X_{\text{safe}, i}$ = safe threshold capacity of Cell $i$
- $\lambda_2$ = safe-threshold penalty weight

This penalty activates when a cell is above the desired safe storage level, even if it is still below physical capacity.

In [69]:
# 9. Safe occupancy threshold setup for 8-cell CTM
# Purpose:Define the safe occupancy threshold for each CTM cell.

#safe threshold multiple . same value used for all the set ups
eta = 0.7
safe_threshold_capacity = {}

for cell in N_max_value:
    safe_threshold_capacity[cell] = eta * N_max_value[cell]


safe_threshold_capacity_df = pd.DataFrame({
    "cell": list(safe_threshold_capacity.keys()),
    "eta": [eta for _ in safe_threshold_capacity],
    "physical_capacity": [
        N_max_value[cell]
        for cell in safe_threshold_capacity
    ],
    "safe_threshold_capacity": [
        safe_threshold_capacity[cell]
        for cell in safe_threshold_capacity
    ],
})

print(" Safe Threshold Capacity")
display(safe_threshold_capacity_df.round(3))

 Safe Threshold Capacity


,cell,eta,physical_capacity,safe_threshold_capacity
0,Cell 1,0.7,591.062,413.744
1,Cell 2,0.7,591.062,413.744
2,Cell 3,0.7,591.062,413.744
3,Cell 4,0.7,591.062,413.744
4,Cell 5,0.7,709.275,496.492
5,Cell 6,0.7,591.062,413.744
6,Cell 7,0.7,709.275,496.492
7,Cell 8,0.7,591.062,413.744


### Capacity Penalty and 30-Second CTM State Update

## Purpose

This section defines two core pieces of the official 8-cell, 30-second state-based CTM benchmark:

1. the capacity penalty function
2. the CTM one-step state update function

The capacity penalty discourages unsafe, physically unrealistic, or spillback-heavy traffic states. The CTM update function advances the mainline vehicle state by one 30-second timestep using sending and receiving logic.

The official benchmark uses the same capacity penalty structure as the ADMM and ADMM-MPC models.

## Capacity Penalty Components

The total capacity penalty has four components:

$$
L_{\text{cap}}
=
L_{\text{doorway}}
+
L_{\text{safe}}
+
L_{\text{physical}}
+
L_{\text{spillback}}
$$

The penalty weights used in this benchmark are:

$$
\lambda_1 = 1.0
$$

$$
\lambda_2 = 0.5
$$

$$
\lambda_3 = 1.0
$$

$$
\lambda_4 = 0.5
$$

where:

- $\lambda_1$ = doorway capacity penalty weight
- $\lambda_2$ = safe-threshold penalty weight
- $\lambda_3$ = physical-capacity penalty weight
- $\lambda_4$ = ramp spillback penalty weight

## 1. Doorway Capacity Penalty

The doorway capacity penalty checks whether the total inflow entering a cell exceeds the cell doorway capacity.

For each cell:

$$
L_{\text{doorway}, i}(t)
=
\lambda_1
\left[
\max\left(q_{\text{in}, i}(t) + u_{\text{in}, i}(t) - C_i, 0\right)
\right]^2
$$

where:

- $q_{\text{in}, i}(t)$ = mainline inflow into Cell $i$
- $u_{\text{in}, i}(t)$ = on-ramp inflow entering Cell $i$
- $C_i$ = doorway capacity of Cell $i$

This penalty activates when the combined mainline inflow and ramp inflow into a cell exceed the cell doorway capacity.

## 2. Safe-Threshold Penalty

The safe-threshold penalty checks whether the next CTM state exceeds the desired safe occupancy level.

For each cell:

$$
L_{\text{safe}, i}(t)
=
\lambda_2
\left[
\max\left(x_i(t+1) - X_{\text{safe}, i}, 0\right)
\right]^2
$$

where:

- $x_i(t+1)$ = next vehicle state of Cell $i$
- $X_{\text{safe}, i}$ = safe occupancy threshold of Cell $i$

This penalty can activate even when the cell is still below physical capacity. It represents operation above the desired safe storage level.

## 3. Physical Capacity Penalty

The physical capacity penalty checks whether the next CTM state exceeds the physical storage capacity of the cell.

For each cell:

$$
L_{\text{physical}, i}(t)
=
\lambda_3
\left[
\max\left(x_i(t+1) - N^{\max}_i, 0\right)
\right]^2
$$

where:

- $N^{\max}_i$ = physical vehicle capacity of Cell $i$

This penalty activates only when the cell state exceeds the maximum physical storage capacity.

## 4. Spillback Penalty

The spillback penalty checks whether a ramp queue exceeds the maximum ramp storage capacity.

For each ramp:

$$
L_{\text{spillback}, r}(t)
=
\lambda_4
\left[
\text{spillback}_r(t)
\right]^2
$$

where:

- $\text{spillback}_r(t)$ = number of vehicles exceeding the storage capacity of ramp $r$

## Total Capacity Penalty

At one CTM timestep, the total capacity penalty is:

$$
L_{\text{cap}}(t)
=
\sum_i L_{\text{doorway}, i}(t)
+
\sum_i L_{\text{safe}, i}(t)
+
\sum_i L_{\text{physical}, i}(t)
+
\sum_r L_{\text{spillback}, r}(t)
$$

This penalty is calculated at every 30-second CTM step and then summed over the full 120-step benchmark horizon.

## CTM 30-Second State Update

The CTM update function advances the 8-cell mainline state by one 30-second timestep.

For each cell, the sending flow is:

$$
S_i(t) = \min\left(x_i(t), C_i\right)
$$

where:

- $S_i(t)$ = sending flow from Cell $i$
- $x_i(t)$ = current vehicle state of Cell $i$
- $C_i$ = doorway capacity of Cell $i$

The receiving capacity is:

$$
R_i(t)
=
\max\left(
0,
\min\left(C_i, N^{\max}_i - x_i(t)\right)
\right)
$$

where:

- $R_i(t)$ = receiving capacity of Cell $i$
- $N^{\max}_i$ = physical capacity of Cell $i$

For Cells 1 through 7, the mainline outflow is limited by upstream sending and downstream receiving:

$$
q_{\text{out}, i}(t)
=
\min\left(S_i(t), R_{i+1}(t)\right)
$$

For Cell 8, the outflow exits the corridor:

$$
q_{\text{out}, 8}(t) = S_8(t)
$$

The inflow into Cell 1 is the PeMS boundary inflow:

$$
q_{\text{in}, 1}(t) = q_{\text{boundary}}(t)
$$

For Cells 2 through 8, inflow comes from the upstream cell outflow:

$$
q_{\text{in}, i}(t) = q_{\text{out}, i-1}(t)
$$

The next CTM state is:

$$
x_i(t+1)
=
x_i(t)
+
q_{\text{in}, i}(t)
+
u_{\text{in}, i}(t)
-
q_{\text{out}, i}(t)
-
f_{\text{out}, i}(t)
$$

where:

- $u_{\text{in}, i}(t)$ = on-ramp inflow entering Cell $i$
- $f_{\text{out}, i}(t)$ = off-ramp outflow leaving Cell $i$

Finally, numerical safety is applied so that vehicle count cannot be negative:

$$
x_i(t+1) = \max\left(x_i(t+1), 0\right)
$$

## First-Step CTM Check

The first-step check verifies that the CTM update function runs correctly using the first 30-second benchmark input values.

This check starts from an empty mainline state and applies:

- first-step boundary inflow
- first-step observed on-ramp releases
- first-step off-ramp flows
- doorway capacity
- physical capacity

The output table shows the resulting one-step values for:

- current state
- mainline inflow
- ramp inflow
- mainline outflow
- off-ramp outflow
- next state

The old synthetic 6-segment capacity test is not used in the final benchmark. The official benchmark uses the actual 8-cell CTM states, flows, ramp queues, and spillback values generated during the state-based simulation.

In [70]:
# 10. Capacity penalty function for 8-cell / 30-sec CTM
# Purpose: Compute the capacity-related penalties used in the official state-based benchmark, ADMM, and ADMM-MPC.

# Penalty weight
lambda_1 = 1.0   # doorway capacity penalty weight
lambda_2 = 0.5   # safe threshold penalty weight
lambda_3 = 1.0   # physical capacity penalty weight
lambda_4 = 0.5   # spillback penalty weight


def capacity_penalty_one_step(
    q_in,
    u_in_step,
    x_next,
    doorway_capacity,
    safe_threshold_capacity,
    physical_capacity,
    spillback_by_ramp,
    lambda_1,
    lambda_2,
    lambda_3,
    lambda_4
):
    doorway_penalty_by_cell = {}
    safe_threshold_penalty_by_cell = {}
    physical_capacity_penalty_by_cell = {}
    spillback_penalty_by_ramp = {}

    total_doorway_penalty = 0.0
    total_safe_threshold_penalty = 0.0
    total_physical_capacity_penalty = 0.0
    total_spillback_penalty = 0.0



    # 1.  doorway, safe threshold, and physical capacity penalties
    for cell in x_next:
        # Doorway capacity penalty
        doorway_overflow = max(
            q_in[cell] + u_in_step[cell] - doorway_capacity[cell],
            0.0
        )
        doorway_penalty = lambda_1 * (doorway_overflow ** 2)

        doorway_penalty_by_cell[cell] = doorway_penalty
        total_doorway_penalty += doorway_penalty


        # Safe threshold penalty
        safe_overflow = max(
            x_next[cell] - safe_threshold_capacity[cell],
            0.0
        )

        safe_threshold_penalty = lambda_2 * (safe_overflow ** 2)

        safe_threshold_penalty_by_cell[cell] = safe_threshold_penalty
        total_safe_threshold_penalty += safe_threshold_penalty


        # Physical capacity penalty
        physical_overflow = max(
            x_next[cell] - physical_capacity[cell],
            0.0
        )

        physical_capacity_penalty = lambda_3 * (physical_overflow ** 2)

        physical_capacity_penalty_by_cell[cell] = physical_capacity_penalty
        total_physical_capacity_penalty += physical_capacity_penalty


    # 2. Ramp spillback penalty
    for ramp in spillback_by_ramp:

        spillback_penalty = lambda_4 * (
            spillback_by_ramp[ramp] ** 2
        )

        spillback_penalty_by_ramp[ramp] = spillback_penalty
        total_spillback_penalty += spillback_penalty
  # 3. Total capacity penalt
    total_capacity_penalty = (
        total_doorway_penalty
        + total_safe_threshold_penalty
        + total_physical_capacity_penalty
        + total_spillback_penalty
    )
    return {
        "doorway_penalty_by_cell": doorway_penalty_by_cell,
        "safe_threshold_penalty_by_cell": safe_threshold_penalty_by_cell,
        "physical_capacity_penalty_by_cell": physical_capacity_penalty_by_cell,
        "spillback_penalty_by_ramp": spillback_penalty_by_ramp,

        "total_doorway_penalty": total_doorway_penalty,
        "total_safe_threshold_penalty": total_safe_threshold_penalty,
        "total_physical_capacity_penalty": total_physical_capacity_penalty,
        "total_spillback_penalty": total_spillback_penalty,
        "total_capacity_penalty": total_capacity_penalty,
    }


In [71]:
# Replace ctm_30sec_step with conservative off-ramp treatment
def ctm_30sec_step(
    x_current,
    q_in_boundary_step,
    u_in_step,
    f_out_step,
    doorway_capacity,
    physical_capacity
):
    sending = {}
    receiving = {}
    q_out = {}
    q_in = {}
    actual_f_out = {}
    x_next = {}

    cells = [f"Cell {i}" for i in range(1, 9)]


    # 1. Sending flow
    for cell in cells:
        sending[cell] = min(
            x_current[cell],
            doorway_capacity[cell]
        )

    # 2. Receiving flow
    for cell in cells:
        receiving[cell] = max(
            0.0,
            min(
                doorway_capacity[cell],
                physical_capacity[cell] - x_current[cell]
            )
        )

    # 3. Mainline outflow
    for i in range(len(cells)):

        current_cell = cells[i]

        if i < len(cells) - 1:
            downstream_cell = cells[i + 1]

            q_out[current_cell] = min(
                sending[current_cell],
                receiving[downstream_cell]
            )

        else:
            q_out[current_cell] = sending[current_cell]

    # 4. Mainline inflow
    q_in["Cell 1"] = q_in_boundary_step

    for i in range(1, len(cells)):
        current_cell = cells[i]
        upstream_cell = cells[i - 1]

        q_in[current_cell] = q_out[upstream_cell]


    # 5. Conservative off-ramp outflow and CTM update
    for cell in cells:

        available_after_mainline_outflow = max(
            0.0,
            x_current[cell]
            + q_in[cell]
            + u_in_step[cell]
            - q_out[cell]
        )

        actual_f_out[cell] = min(
            f_out_step[cell],
            available_after_mainline_outflow
        )

        x_next[cell] = (
            x_current[cell]
            + q_in[cell]
            + u_in_step[cell]
            - q_out[cell]
            - actual_f_out[cell]
        )

        # Numerical safety only. This should not activate except
        # for tiny floating-point noise.
        x_next[cell] = max(x_next[cell], 0.0)

    return x_next, q_out, q_in, sending, receiving, actual_f_out

In [72]:
# CTM first-step state update check
# Purpose: Verify that ctm_30sec_step runs using the first 30-second input.

ctm_test_x_current = {
    "Cell 1": 0.0,
    "Cell 2": 0.0,
    "Cell 3": 0.0,
    "Cell 4": 0.0,
    "Cell 5": 0.0,
    "Cell 6": 0.0,
    "Cell 7": 0.0,
    "Cell 8": 0.0,
}


ctm_test_observed_release_step = {
    "u1": observed_release_series["u1"][0],
    "u2": observed_release_series["u2"][0],
    "u3": observed_release_series["u3"][0],
    "u4": observed_release_series["u4"][0],
    "u5": observed_release_series["u5"][0],
}


ctm_test_u_in_step = {
    "Cell 1": 0.0,
    "Cell 2": ctm_test_observed_release_step["u1"],
    "Cell 3": 0.0,
    "Cell 4": ctm_test_observed_release_step["u2"],
    "Cell 5": ctm_test_observed_release_step["u3"],
    "Cell 6": ctm_test_observed_release_step["u4"],
    "Cell 7": ctm_test_observed_release_step["u5"],
    "Cell 8": 0.0,
}


ctm_test_x_next, ctm_test_q_out, ctm_test_q_in, ctm_test_sending, ctm_test_receiving, ctm_test_actual_f_out = ctm_30sec_step( ctm_test_x_current,
    q_in_boundary_series[0],
    ctm_test_u_in_step,
    f_out_series[0],
    doorway_capacity,
    physical_capacity
)


ctm_first_step_check_df = pd.DataFrame({
    "cell": list(ctm_test_x_next.keys()),
    "x_current": [
        ctm_test_x_current[cell]
        for cell in ctm_test_x_next
    ],
    "q_in": [
        ctm_test_q_in[cell]
        for cell in ctm_test_x_next
    ],
    "u_in": [
        ctm_test_u_in_step[cell]
        for cell in ctm_test_x_next
    ],
    "q_out": [
        ctm_test_q_out[cell]
        for cell in ctm_test_x_next
    ],
    "requested_f_out": [
    f_out_series[0][cell]
    for cell in ctm_test_x_next
    ],
    "actual_f_out": [
    ctm_test_actual_f_out[cell]
    for cell in ctm_test_x_next
    ],
    "f_out_capped_gap": [
    f_out_series[0][cell] - ctm_test_actual_f_out[cell]
    for cell in ctm_test_x_next
    ],
    "x_next": [
        ctm_test_x_next[cell]
        for cell in ctm_test_x_next
    ],
})


print(" CTM First-Step State Update Check ")
display(ctm_first_step_check_df.round(3))

 CTM First-Step State Update Check 


,cell,x_current,q_in,u_in,q_out,requested_f_out,actual_f_out,f_out_capped_gap,x_next
0,Cell 1,0.0,64.5,0.0,0.0,0.0,0.0,0.0,64.5
1,Cell 2,0.0,0.0,8.8,0.0,0.0,0.0,0.0,8.8
2,Cell 3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Cell 4,0.0,0.0,3.9,0.0,0.0,0.0,0.0,3.9
4,Cell 5,0.0,0.0,7.3,0.0,0.0,0.0,0.0,7.3
5,Cell 6,0.0,0.0,5.6,0.0,8.2,5.6,2.6,0.0
6,Cell 7,0.0,0.0,6.4,0.0,0.8,0.8,0.0,5.6
7,Cell 8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 12–14. Official 8-Cell CTM State-Based Mainline Delay Setup

## Purpose

This section prepares the official state-based CTM benchmark used for comparison with ADMM and ADMM-MPC.

Unlike the earlier flow/speed-based sanity check, this benchmark uses the actual CTM state variables:

$$
x_i(t)
$$

where $x_i(t)$ is the number of vehicles stored in CTM Cell $i$ at time step $t$.

The official benchmark uses:

$$
8 \text{ CTM cells}
$$

$$
\Delta t = 0.5 \text{ minutes}
$$

$$
120 \text{ steps}
$$

which corresponds to the one-hour period:

$$
08{:}00 \text{ to } 09{:}00
$$

---

## 12. Real 8-Cell Initial State

The initial state represents the number of vehicles stored in each CTM cell at the beginning of the benchmark window:

$$
x_i(0)
$$

for each cell:

$$
i = 1, 2, \dots, 8
$$

These values are used as the starting point for the CTM simulation.

The benchmark also initializes the ramp queues before the simulation begins. In this benchmark, all initial ramp queues are set to zero:

$$
R_j(0) = 0
$$

for each ramp:

$$
j = 1, 2, \dots, 5
$$

This means the field-observed ramp-release benchmark starts without any pre-existing ramp queue.

## CTM 30-Second State Update

The CTM state update uses sending and receiving logic.

For each cell, the sending flow is:

$$
S_i(t) = \min\left(x_i(t), C_i\right)
$$

where:

- $S_i(t)$ = sending flow from Cell $i$
- $x_i(t)$ = current vehicle state of Cell $i$
- $C_i$ = doorway capacity of Cell $i$

The receiving capacity is:

$$
R_i(t)
=
\max\left(
0,
\min\left(C_i, N^{\max}_i - x_i(t)\right)
\right)
$$

where:

- $R_i(t)$ = receiving capacity of Cell $i$
- $N^{\max}_i$ = physical capacity of Cell $i$

For Cells 1 through 7, mainline outflow is limited by upstream sending and downstream receiving:

$$
q_{\text{out},i}(t)
=
\min\left(S_i(t), R_{i+1}(t)\right)
$$

For Cell 8, outflow exits the corridor:

$$
q_{\text{out},8}(t) = S_8(t)
$$

Cell 1 receives the PeMS boundary inflow:

$$
q_{\text{in},1}(t) = q_{\text{boundary}}(t)
$$

Cells 2 through 8 receive inflow from the upstream cell:

$$
q_{\text{in},i}(t) = q_{\text{out},i-1}(t)
$$

The next CTM state is calculated as:

$$
x_i(t+1)
=
x_i(t)
+
q_{\text{in},i}(t)
+
u_{\text{in},i}(t)
-
q_{\text{out},i}(t)
-
f_{\text{out},i}(t)
$$

where:

- $u_{\text{in},i}(t)$ = on-ramp inflow entering Cell $i$
- $f_{\text{out},i}(t)$ = off-ramp outflow leaving Cell $i$

Finally, vehicle storage is clipped at zero for numerical safety:

$$
x_i(t+1) = \max\left(x_i(t+1), 0\right)
$$

---

## 13. 8-Cell Free-Flow Travel Time Calculation

The CTM delay formula requires a free-flow travel time for each CTM cell:

$$
TT_{\text{ff},i}
$$

Instead of hard-coding these values, this section calculates them from:

- mainline detector postmiles
- detector-to-detector free-flow segment speeds
- the 8 equal CTM cells

The CTM uses 8 equal-length cells, while the physical detector spacing creates 6 detector-to-detector roadway segments. Therefore, one CTM cell may overlap with one or more physical segments.

For each CTM cell, the free-flow travel time is calculated as:

$$
TT_{\text{ff},i}
=
\sum_s
\left(
\frac{L_{i,s}}{v_{\text{ff},s}}
\right)
\cdot 60
$$

where:

- $L_{i,s}$ = overlap length between CTM Cell $i$ and physical segment $s$, measured in miles
- $v_{\text{ff},s}$ = free-flow speed of physical segment $s$, measured in miles per hour
- $60$ = conversion factor from hours to minutes

This produces one free-flow travel time value for each CTM cell:

$$
TT_{\text{ff},1}, TT_{\text{ff},2}, \dots, TT_{\text{ff},8}
$$

---

## 14. State-Based Mainline Delay Calculation

The mainline delay is calculated from CTM states, not directly from observed speed.

For each CTM cell, the total travel time spent inside the cell during one 30-second step is:

$$
TTT_i(t)
=
\frac{x_i(t) + x_i(t+1)}{2}
\cdot
\Delta t
$$

where:

- $x_i(t)$ = number of vehicles in Cell $i$ before the CTM update
- $x_i(t+1)$ = number of vehicles in Cell $i$ after the CTM update
- $\Delta t = 0.5$ minutes

The free-flow component is:

$$
\left(q_{\text{out},i}(t) + f_{\text{out},i}(t)\right)
\cdot
TT_{\text{ff},i}
$$

where:

- $q_{\text{out},i}(t)$ = mainline outflow from Cell $i$
- $f_{\text{out},i}(t)$ = off-ramp flow leaving Cell $i$
- $TT_{\text{ff},i}$ = free-flow travel time for Cell $i$

The raw state-based mainline delay is:

$$
D^{raw}_{M,i}(t)
=
TTT_i(t)
-
\left(q_{\text{out},i}(t) + f_{\text{out},i}(t)\right)
\cdot
TT_{\text{ff},i}
$$

Because delay cannot physically be negative, the official benchmark clamps cell-level delay at zero:

$$
D_{M,i}(t)
=
\max\left(D^{raw}_{M,i}(t), 0\right)
$$

This avoids artificial negative delay from draining cells canceling real positive delay from congested cells.

The total mainline delay for one CTM step is:

$$
D_M(t)
=
\sum_{i=1}^{8}
D_{M,i}(t)
$$

This function is applied over all 120 CTM steps to compute the official state-based benchmark mainline delay.

## First-Step Mainline Delay Check

The first-step check applies the CTM update using:

- the real 08:00 initial mainline state
- the first 30-second boundary inflow
- the first 30-second observed on-ramp releases
- the first 30-second off-ramp flows
- doorway capacity
- physical capacity

The output table reports, for each cell:

- current state
- next state
- mainline outflow
- off-ramp flow
- total travel time
- free-flow component
- clamped mainline delay

The displayed first-step delay confirms that the state-based delay calculation is working with the corrected 8-cell CTM structure.

In [73]:
# 08:00 PeMS-derived mainline initial state
# Units: vehicles in each CTM cell

benchmark_start_timestamp = pd.Timestamp("2026-01-08 08:00:00")

mainline_ids = [
    1201419,
    1201469,
    1201497,
    1201525,
    1201558,
    1201589,
    1201620,
]

# 1. Extract PeMS mainline detector data at 08:00
mainline_8am_state_data = selected_data_morning[
    (selected_data_morning["station_id"].isin(mainline_ids))
    & (selected_data_morning["timestamp"] == benchmark_start_timestamp)
].copy()

mainline_8am_state_data["total_flow"] = pd.to_numeric(
    mainline_8am_state_data["total_flow"],
    errors="coerce",
)

mainline_8am_state_data["avg_speed"] = pd.to_numeric(
    mainline_8am_state_data["avg_speed"],
    errors="coerce",
)

mainline_8am_state_data = mainline_8am_state_data.merge(
    selected_metadata_clean[
        [
            "station_id",
            "station_name",
            "absolute_postmile",
            "lanes",
        ]
    ],
    on="station_id",
    how="left",
)

mainline_8am_state_data = mainline_8am_state_data.sort_values(
    "absolute_postmile"
).reset_index(drop=True)


# Safety checks for initial-state derivation

if len(mainline_8am_state_data) != len(mainline_ids):
    raise ValueError(
        f"Expected {len(mainline_ids)} mainline detectors at 08:00, "
        f"but found {len(mainline_8am_state_data)}."
    )

if mainline_8am_state_data["total_flow"].isna().any():
    raise ValueError("Missing total_flow in 08:00 mainline detector data.")

if mainline_8am_state_data["avg_speed"].isna().any():
    raise ValueError("Missing avg_speed in 08:00 mainline detector data.")

if (mainline_8am_state_data["avg_speed"] <= 0).any():
    raise ValueError("Nonpositive avg_speed found in 08:00 mainline detector data.")
# 2. Convert PeMS flow/speed to detector density

mainline_8am_state_data["flow_vph"] = (
    12.0 * mainline_8am_state_data["total_flow"]
)

mainline_8am_state_data["density_veh_per_mile"] = (
    mainline_8am_state_data["flow_vph"]
    / mainline_8am_state_data["avg_speed"]
)



# 3. Build 8 equal-length CTM cells
corridor_start_postmile = mainline_8am_state_data["absolute_postmile"].min()
corridor_end_postmile = mainline_8am_state_data["absolute_postmile"].max()

num_cells = 8
cell_length = (corridor_end_postmile - corridor_start_postmile) / num_cells

cell_geometry_rows = []

for i in range(1, num_cells + 1):
    cell_start = corridor_start_postmile + (i - 1) * cell_length
    cell_end = corridor_start_postmile + i * cell_length
    cell_midpoint = 0.5 * (cell_start + cell_end)

    cell_geometry_rows.append({
        "cell": f"Cell {i}",
        "cell_start_postmile": cell_start,
        "cell_end_postmile": cell_end,
        "cell_midpoint_postmile": cell_midpoint,
        "cell_length_miles": cell_length,
    })

mainline_cell_geometry_df = pd.DataFrame(cell_geometry_rows)

# 4. Interpolate detector density to CTM cell midpoints
detector_postmiles = mainline_8am_state_data[
    "absolute_postmile"
].to_numpy(dtype=float)

detector_densities = mainline_8am_state_data[
    "density_veh_per_mile"
].to_numpy(dtype=float)

cell_midpoints = mainline_cell_geometry_df[
    "cell_midpoint_postmile"
].to_numpy(dtype=float)

mainline_cell_geometry_df["interpolated_density_veh_per_mile"] = np.interp(
    cell_midpoints,
    detector_postmiles,
    detector_densities,
)

mainline_cell_geometry_df["initial_state_x0"] = (
    mainline_cell_geometry_df["interpolated_density_veh_per_mile"]
    * mainline_cell_geometry_df["cell_length_miles"]
)

# 5. Final benchmark initial state

mainline_initial_state = dict(
    zip(
        mainline_cell_geometry_df["cell"],
        mainline_cell_geometry_df["initial_state_x0"],
    )
)

# 6. Derivation tables

mainline_initial_detector_derivation_df = mainline_8am_state_data[
    [
        "station_id",
        "station_name",
        "absolute_postmile",
        "lanes",
        "total_flow",
        "avg_speed",
        "flow_vph",
        "density_veh_per_mile",
    ]
].copy()

mainline_initial_state_derivation_df = mainline_cell_geometry_df[
    [
        "cell",
        "cell_start_postmile",
        "cell_end_postmile",
        "cell_midpoint_postmile",
        "cell_length_miles",
        "interpolated_density_veh_per_mile",
        "initial_state_x0",
    ]
].copy()

print("PeMS 08:00 detector-level density derivation")
display(mainline_initial_detector_derivation_df.round(3))

print("PeMS-derived 8-cell initial mainline state")
display(mainline_initial_state_derivation_df.round(3))

print("mainline_initial_state:")
for cell, value in mainline_initial_state.items():
    print(cell, "=", round(value, 3))
# Initial ramp queue
# Benchmark starts with zero queue unless you intentionally
# choose a nonzero fill ratio.
 
ramp_queue_0 = {
    "u1": 0.0,
    "u2": 0.0,
    "u3": 0.0,
    "u4": 0.0,
    "u5": 0.0,
}


# Clean check
mainline_initial_state_df = pd.DataFrame({
    "cell": list(mainline_initial_state.keys()),
    "initial_state_x0": list(mainline_initial_state.values()),
    "safe_threshold_capacity": [
        safe_threshold_capacity[cell]
        for cell in mainline_initial_state
    ],
    "physical_capacity": [
        physical_capacity[cell]
        for cell in mainline_initial_state
    ],
})

ramp_queue_0_df = pd.DataFrame({
    "ramp": list(ramp_queue_0.keys()),
    "initial_queue": list(ramp_queue_0.values()),
    "max_queue": [
        ramp_max_queue_by_u[ramp]
        for ramp in ramp_queue_0
    ],
})

print("8-Cell Mainline Initial State")
display(mainline_initial_state_df.round(3))

print("Initial Ramp Queue ")
display(ramp_queue_0_df.round(3))

PeMS 08:00 detector-level density derivation


,station_id,station_name,absolute_postmile,lanes,total_flow,avg_speed,flow_vph,density_veh_per_mile
0,1201419,RED HILL,8.17,5,645.0,39.4,7740.0,196.447
1,1201469,BRISTOL 1,9.31,5,681.0,40.0,8172.0,204.300
2,1201497,FAIRVIEW,10.05,5,622.0,69.5,7464.0,107.396
3,1201525,HARBOR 1,10.97,6,737.0,30.6,8844.0,289.020
4,1201558,HARBOR 2,11.27,5,599.0,27.7,7188.0,259.495
5,1201589,EUCLID,12.27,6,680.0,22.3,8160.0,365.919
6,1201620,TALBERT,13.07,5,550.0,33.7,6600.0,195.846


PeMS-derived 8-cell initial mainline state


,cell,cell_start_postmile,cell_end_postmile,cell_midpoint_postmile,cell_length_miles,interpolated_density_veh_per_mile,initial_state_x0
0,Cell 1,8.170,8.782,8.476,0.612,198.556,121.616
1,Cell 2,8.782,9.395,9.089,0.612,202.776,124.200
2,Cell 3,9.395,10.008,9.701,0.612,153.065,93.752
3,Cell 4,10.008,10.620,10.314,0.612,159.464,97.672
4,Cell 5,10.620,11.232,10.926,0.612,280.383,171.734
5,Cell 6,11.232,11.845,11.539,0.612,288.096,176.459
6,Cell 7,11.845,12.458,12.151,0.612,353.281,216.385
7,Cell 8,12.458,13.070,12.764,0.612,260.952,159.833


mainline_initial_state:
Cell 1 = 121.616
Cell 2 = 124.2
Cell 3 = 93.752
Cell 4 = 97.672
Cell 5 = 171.734
Cell 6 = 176.459
Cell 7 = 216.385
Cell 8 = 159.833
8-Cell Mainline Initial State


,cell,initial_state_x0,safe_threshold_capacity,physical_capacity
0,Cell 1,121.616,413.744,591.062
1,Cell 2,124.200,413.744,591.062
2,Cell 3,93.752,413.744,591.062
3,Cell 4,97.672,413.744,591.062
4,Cell 5,171.734,496.492,709.275
5,Cell 6,176.459,413.744,591.062
6,Cell 7,216.385,496.492,709.275
7,Cell 8,159.833,413.744,591.062


Initial Ramp Queue 


,ramp,initial_queue,max_queue
0,u1,0.0,28.669
1,u2,0.0,72.356
2,u3,0.0,56.168
3,u4,0.0,72.441
4,u5,0.0,48.819


In [74]:
# 13. Calculate 8-cell free-flow travel time
# Purpose: Convert the 6 detector-to-detector free-flow segments into 8 equal CTM-cell free-flow travel times.

# Mainline detector postmiles
mainline_station_postmile = (
    selected_metadata_clean[
        selected_metadata_clean["station_id"].isin(mainline_ids)
    ][
        ["station_id", "station_name", "absolute_postmile"]
    ]
    .sort_values("absolute_postmile")
    .reset_index(drop=True)
)


# Physical detector-to-detector segment free-flow speeds

physical_segment_speed_df = pd.DataFrame({
    "physical_segment": [
        "RED HILL → BRISTOL 1",
        "BRISTOL 1 → FAIRVIEW",
        "FAIRVIEW → HARBOR 1",
        "HARBOR 1 → HARBOR 2",
        "HARBOR 2 → EUCLID",
        "EUCLID → TALBERT",
    ],
    "start_postmile": [
        mainline_station_postmile.loc[0, "absolute_postmile"],
        mainline_station_postmile.loc[1, "absolute_postmile"],
        mainline_station_postmile.loc[2, "absolute_postmile"],
        mainline_station_postmile.loc[3, "absolute_postmile"],
        mainline_station_postmile.loc[4, "absolute_postmile"],
        mainline_station_postmile.loc[5, "absolute_postmile"],
    ],
    "end_postmile": [
        mainline_station_postmile.loc[1, "absolute_postmile"],
        mainline_station_postmile.loc[2, "absolute_postmile"],
        mainline_station_postmile.loc[3, "absolute_postmile"],
        mainline_station_postmile.loc[4, "absolute_postmile"],
        mainline_station_postmile.loc[5, "absolute_postmile"],
        mainline_station_postmile.loc[6, "absolute_postmile"],
    ],
    "v_ff_mph": [
        v_ff_1,
        v_ff_2,
        v_ff_3,
        v_ff_4,
        v_ff_5,
        v_ff_6,
    ],
})


# Build 8 equal CTM cells over full corridor

corridor_start_postmile = mainline_station_postmile["absolute_postmile"].min()
corridor_end_postmile = mainline_station_postmile["absolute_postmile"].max()

num_cells = 8
cell_length = (corridor_end_postmile - corridor_start_postmile) / num_cells


ctm_cell_df = pd.DataFrame({
    "cell": [f"Cell {i}" for i in range(1, num_cells + 1)],
    "cell_start_postmile": [
        corridor_start_postmile + (i - 1) * cell_length
        for i in range(1, num_cells + 1)
    ],
    "cell_end_postmile": [
        corridor_start_postmile + i * cell_length
        for i in range(1, num_cells + 1)
    ],
})


# Helper: overlap length between one CTM cell and one physical segment

def overlap_length(cell_start, cell_end, segment_start, segment_end):
    overlap = max(
        0.0,
        min(cell_end, segment_end) - max(cell_start, segment_start)
    )

    return overlap


# Calculate TT_ff for each CTM cell

tt_ff_min = {}
tt_ff_rows = []

for _, cell_row in ctm_cell_df.iterrows():

    cell = cell_row["cell"]
    cell_start = cell_row["cell_start_postmile"]
    cell_end = cell_row["cell_end_postmile"]

    cell_tt_ff_min = 0.0

    for _, segment_row in physical_segment_speed_df.iterrows():

        segment_start = segment_row["start_postmile"]
        segment_end = segment_row["end_postmile"]
        segment_speed = segment_row["v_ff_mph"]

        overlap = overlap_length(
            cell_start,
            cell_end,
            segment_start,
            segment_end
        )

        if overlap > 0:
            overlap_tt_min = (overlap / segment_speed) * 60.0
            cell_tt_ff_min += overlap_tt_min

    tt_ff_min[cell] = cell_tt_ff_min

    tt_ff_rows.append({
        "cell": cell,
        "cell_start_postmile": cell_start,
        "cell_end_postmile": cell_end,
        "cell_length_miles": cell_end - cell_start,
        "TT_ff_min": cell_tt_ff_min,
    })


tt_ff_min_df = pd.DataFrame(tt_ff_rows)


print("Calculated 8-Cell Free-Flow Travel Time ")
display(tt_ff_min_df.round(3))

print("Calculated tt_ff_min dictionary:")
for cell in tt_ff_min:
    print(cell, ":", round(tt_ff_min[cell], 3))


Calculated 8-Cell Free-Flow Travel Time 


,cell,cell_start_postmile,cell_end_postmile,cell_length_miles,TT_ff_min
0,Cell 1,8.170,8.782,0.613,0.567
1,Cell 2,8.782,9.395,0.612,0.564
2,Cell 3,9.395,10.008,0.613,0.544
3,Cell 4,10.008,10.620,0.613,0.533
4,Cell 5,10.620,11.232,0.612,0.534
5,Cell 6,11.232,11.845,0.613,0.537
6,Cell 7,11.845,12.458,0.612,0.538
7,Cell 8,12.458,13.070,0.613,0.540


Calculated tt_ff_min dictionary:
Cell 1 : 0.567
Cell 2 : 0.564
Cell 3 : 0.544
Cell 4 : 0.533
Cell 5 : 0.534
Cell 6 : 0.537
Cell 7 : 0.538
Cell 8 : 0.54


In [75]:
# 14. Mainline delay setup for 8-cell CTM
delta_t = 0.5  # 30 seconds = 0.5 minutes

# Build CTM ramp inflow dictionary from actual ramp releases
def build_u_in_from_ramp_release(ramp_release_step):
    u_in_step = {
        "Cell 1": 0.0,
        "Cell 2": ramp_release_step["u1"],  # Bristol 1
        "Cell 3": 0.0,
        "Cell 4": ramp_release_step["u2"],  # Fairview
        "Cell 5": ramp_release_step["u3"],  # Harbor 1
        "Cell 6": ramp_release_step["u4"],  # Harbor 2
        "Cell 7": ramp_release_step["u5"],  # Euclid
        "Cell 8": 0.0,
    }

    return u_in_step


# Mainline delay for one 30-second CTM step

def mainline_delay_one_step(
    x_now,
    x_next,
    q_out,
    actual_f_out,
    tt_ff_min,
    delta_t
):
    rows = []
    total_mainline_delay = 0.0

    for cell in x_now:

        # Actual total time spent in cell during this CTM step
        ttt = ((x_now[cell] + x_next[cell]) / 2.0) * delta_t

        # Free-flow time for vehicles leaving through mainline/out-ramp
        ff_term = (q_out[cell] + actual_f_out[cell]) * tt_ff_min[cell]

        # State-based mainline delay
        # If delay is positive, keep it.
        # If delay is negative, set it to 0.
        delay_raw = ttt - ff_term
        delay = max(delay_raw, 0.0)
        total_mainline_delay += delay

        rows.append({
            "Cell": cell,
            "x_now": x_now[cell],
            "x_next": x_next[cell],
            "q_out": q_out[cell],
            "actual_f_out": actual_f_out[cell],
            "TTT_veh_min": ttt,
            "free_flow_component": ff_term,
            "mainline_delay_veh_min": delay,
        })

    mainline_delay_df = pd.DataFrame(rows)

    return mainline_delay_df, total_mainline_delay


# first-step check using actual 08:00 initial state
observed_release_step_0 = {
    "u1": observed_release_series["u1"][0],
    "u2": observed_release_series["u2"][0],
    "u3": observed_release_series["u3"][0],
    "u4": observed_release_series["u4"][0],
    "u5": observed_release_series["u5"][0],
}

ramp_arrival_step_0 = {
    "u1": ramp_arrival_series["u1"][0],
    "u2": ramp_arrival_series["u2"][0],
    "u3": ramp_arrival_series["u3"][0],
    "u4": ramp_arrival_series["u4"][0],
    "u5": ramp_arrival_series["u5"][0],
}


B_step_0 = {
    ramp: 0.0
    for ramp in ramp_queue_0
}

(
    R_next_step_0,
    B_next_step_0,
    spillback_step_0,
    actual_release_step_0,
) = ramp_next_queue_with_spillback(
    ramp_queue_0,
    B_step_0,
    ramp_arrival_step_0,
    observed_release_step_0,
    ramp_max_queue_by_u
)

u_in_step_0 = build_u_in_from_ramp_release(
    actual_release_step_0
)

f_out_step_0 = f_out_series[0]

x_next_0, q_out_0, q_in_0, sending_0, receiving_0, actual_f_out_0 = ctm_30sec_step(
    mainline_initial_state,
    q_in_boundary_series[0],
    u_in_step_0,
    f_out_step_0,
    doorway_capacity,
    physical_capacity
)

mainline_delay_df_0, total_mainline_delay_0 = mainline_delay_one_step(
    mainline_initial_state,
    x_next_0,
    q_out_0,
    actual_f_out_0,
    tt_ff_min,
    delta_t
)

print(" First-Step CTM Mainline Delay Check ")
display(mainline_delay_df_0.round(3))

print("Total first-step mainline delay:", round(total_mainline_delay_0, 3), "veh-min")


 First-Step CTM Mainline Delay Check 


,Cell,x_now,x_next,q_out,actual_f_out,TTT_veh_min,free_flow_component,mainline_delay_veh_min
0,Cell 1,121.616,101.789,84.327,0.0,55.851,47.846,8.005
1,Cell 2,124.200,133.000,84.327,0.0,64.300,47.575,16.725
2,Cell 3,93.752,93.752,84.327,0.0,46.876,45.894,0.982
3,Cell 4,97.672,101.572,84.327,0.0,49.811,44.910,4.901
4,Cell 5,171.734,179.034,84.327,0.0,87.692,44.996,42.696
5,Cell 6,176.459,173.859,84.327,8.2,87.579,49.724,37.855
6,Cell 7,216.385,221.985,84.327,0.8,109.592,45.832,63.761
7,Cell 8,159.833,159.833,84.327,0.0,79.917,45.574,34.343


Total first-step mainline delay: 209.268 veh-min


### Local Ramp Delay Calculation

## Purpose

The local ramp delay measures the amount of delay experienced by vehicles waiting in the on-ramp queues.

For each ramp, the queue state is represented as:

$$
R_j(t)
$$

where $R_j(t)$ is the number of vehicles waiting on ramp $j$ at time step $t$.

## Ramp Queue Update

The ramp queue is updated using the current queue, assumed ramp arrival demand, and observed ramp release:

$$
R^{raw}_j(t+1)
=
R_j(t) + a_j(t) - u_j(t)
$$

where:

- $R_j(t)$ = current queue on ramp $j$
- $a_j(t)$ = assumed ramp arrival demand during the 30-second step
- $u_j(t)$ = observed ramp release from PeMS during the 30-second step
- $R^{raw}_j(t+1)$ = raw next queue before applying non-negativity and storage limits

The queue is then constrained so it cannot be negative:

$$
R^{uncapped}_j(t+1)
=
\max\left(R^{raw}_j(t+1), 0\right)
$$

If the uncapped queue exceeds the maximum ramp storage capacity, the stored queue is capped at:

$$
R_j(t+1)
=
\min\left(R^{uncapped}_j(t+1), R_{max,j}\right)
$$

Any excess above $R_{max,j}$ is handled separately through the spillback penalty.

## Local Ramp Delay Formula

Because delay depends on how many vehicles are waiting during the time interval, the average queue is used:

$$
\bar{R}_j(t)
=
\frac{R_j(t) + R_j(t+1)}{2}
$$

The local ramp delay for one ramp during one CTM step is:

$$
D_{L,j}(t)
=
\bar{R}_j(t) \cdot \Delta t
$$

or equivalently:

$$
D_{L,j}(t)
=
\frac{R_j(t) + R_j(t+1)}{2}
\cdot
\Delta t
$$

where:

$$
\Delta t = 0.5 \text{ minutes}
$$

The delay is measured in vehicle-minutes.

## Total Local Ramp Delay

The total local ramp delay for one CTM step is the sum over all five ramps:

$$
D_L(t)
=
\sum_{j=1}^{5}
D_{L,j}(t)
$$

Over the full 120-step benchmark horizon, the total local ramp delay is:

$$
D_L
=
\sum_{t=0}^{119}
\sum_{j=1}^{5}
D_{L,j}(t)
$$

This local delay term is included in the same objective structure used by the benchmark, ADMM, and ADMM-MPC models.

## First-Step Local Delay Check

The first-step check calculates local ramp delay using:

- initial ramp queues
- first-step ramp arrivals
- first-step observed ramp releases
- first-step capped next ramp queues

At the first step, the initial ramp queues are zero. Since the assumed ramp arrival is larger than the observed release, the queues increase slightly. The delay is calculated from the average of the initial and next queue values during the 30-second interval.

The function `ramp_delay_with_cap(...)` uses the capped next ramp queue. Vehicles above the ramp storage limit are handled separately by the spillback penalty.

In [76]:
# 15. Local ramp delay calculation for one CTM step
def ramp_delay_with_cap(
    R_current,
    R_next,
    B_current,
    B_next,
    ramp_arrival_step,
    observed_release_step,
    actual_release_step,
    delta_t
):
    rows = []
    total_local_delay = 0.0

    for ramp in R_current:

        waiting_current = (
            float(R_current[ramp])
            + float(B_current.get(ramp, 0.0))
        )

        waiting_next = (
            float(R_next[ramp])
            + float(B_next.get(ramp, 0.0))
        )

        # Average total waiting demand during this 30-second step
        waiting_avg = (waiting_current + waiting_next) / 2.0

        # Local ramp delay including external spillback/backlog delay
        local_delay = waiting_avg * delta_t

        total_local_delay += local_delay

        rows.append({
            "ramp": ramp,
            "R_current": R_current[ramp],
            "B_current": B_current.get(ramp, 0.0),
            "arrival": ramp_arrival_step[ramp],
            "commanded_release": observed_release_step[ramp],
            "actual_release": actual_release_step[ramp],
            "R_next": R_next[ramp],
            "B_next": B_next.get(ramp, 0.0),
            "waiting_current": waiting_current,
            "waiting_next": waiting_next,
            "waiting_avg": waiting_avg,
            "local_delay_veh_min": local_delay,
        })

    ramp_delay_df = pd.DataFrame(rows)

    return ramp_delay_df, total_local_delay


# First-step local delay check
ramp_delay_df_0, total_local_delay_0 = ramp_delay_with_cap(
    ramp_queue_0,
    R_next_test,
    B_current_test,
    B_next_test,
    ramp_arrival_step_test,
    observed_release_step_test,
    actual_release_step_test,
    delta_t
)

print("First-Step Local Ramp Delay Check")
display(ramp_delay_df_0.round(3))

print("Total first-step local ramp delay:", round(total_local_delay_0, 3), "veh-min")


First-Step Local Ramp Delay Check


,ramp,R_current,B_current,arrival,commanded_release,actual_release,R_next,B_next,waiting_current,waiting_next,waiting_avg,local_delay_veh_min
0,u1,0.0,0.0,13.20,8.8,8.8,4.40,0.0,0.0,4.40,2.200,1.100
1,u2,0.0,0.0,5.85,3.9,3.9,1.95,0.0,0.0,1.95,0.975,0.487
2,u3,0.0,0.0,10.95,7.3,7.3,3.65,0.0,0.0,3.65,1.825,0.912
3,u4,0.0,0.0,8.40,5.6,5.6,2.80,0.0,0.0,2.80,1.400,0.700
4,u5,0.0,0.0,9.60,6.4,6.4,3.20,0.0,0.0,3.20,1.600,0.800


Total first-step local ramp delay: 4.0 veh-min


## 16–17. Official 120-Step State-Based CTM Benchmark and Summary

## Purpose

This section runs the official state-based CTM benchmark for the 08:00–09:00 peak-hour window.

The benchmark uses:

$$
8 \text{ CTM cells}
$$

$$
\Delta t = 0.5 \text{ minutes}
$$

$$
120 \text{ steps}
$$

because:

$$
60 \text{ minutes} \div 0.5 \text{ minutes per step} = 120 \text{ steps}
$$

This benchmark represents the field-observed ramp-release baseline. Ramp releases are not optimized. Instead, the observed PeMS on-ramp release flows are used directly:

$$
u_j(t) = u^{obs}_j(t)
$$

where:

- $u_j(t)$ = ramp release from ramp $j$ at time step $t$
- $u^{obs}_j(t)$ = observed PeMS ramp release from ramp $j$ at time step $t$

The benchmark is compared directly against ADMM and ADMM-MPC because all models use the same CTM structure, inputs, delay formulas, fairness penalty, and capacity penalty definitions.

---

## 16. Official 120-Step State-Based CTM Benchmark Simulation

At each 30-second step, the simulation updates:

$$
x_i(t)
$$

the number of vehicles stored in CTM Cell $i$, and:

$$
R_j(t)
$$

the queue length on ramp $j$.

The simulation uses dynamic 30-second input series built from the 08:00–09:00 PeMS data:

- boundary mainline inflow
- observed on-ramp releases
- assumed ramp arrival demand
- off-ramp flows

## Mainline CTM State Update

The mainline state evolves according to:

$$
x_i(t+1)
=
x_i(t)
+
q_{\text{in},i}(t)
+
u_{\text{in},i}(t)
-
q_{\text{out},i}(t)
-
f_{\text{out},i}(t)
$$

where:

- $x_i(t)$ = vehicle storage in Cell $i$ before the update
- $x_i(t+1)$ = vehicle storage in Cell $i$ after the update
- $q_{\text{in},i}(t)$ = mainline inflow into Cell $i$
- $u_{\text{in},i}(t)$ = on-ramp inflow entering Cell $i$
- $q_{\text{out},i}(t)$ = mainline outflow from Cell $i$
- $f_{\text{out},i}(t)$ = off-ramp flow leaving Cell $i$

The CTM update uses sending and receiving constraints based on doorway capacity and physical capacity.

## Ramp Queue Update

Ramp queues are updated using assumed ramp arrivals and observed ramp releases:

$$
R^{raw}_j(t+1)
=
R_j(t)
+
a_j(t)
-
u^{obs}_j(t)
$$

where:

- $R_j(t)$ = ramp queue before the update
- $a_j(t)$ = assumed ramp arrival demand
- $u^{obs}_j(t)$ = observed PeMS ramp release
- $R^{raw}_j(t+1)$ = raw next ramp queue before applying queue constraints

The queue is constrained to be nonnegative:

$$
R^{uncapped}_j(t+1)
=
\max\left(R^{raw}_j(t+1), 0\right)
$$

If the uncapped queue exceeds the maximum ramp storage capacity, the excess is counted as spillback:

$$
\text{spillback}_j(t)
=
\max\left(R^{uncapped}_j(t+1) - R_{\max,j}, 0\right)
$$

The stored ramp queue is capped at the maximum ramp capacity:

$$
R_j(t+1)
=
\min\left(R^{uncapped}_j(t+1), R_{\max,j}\right)
$$

---

## Objective Terms Computed at Each Step

At every CTM step, the benchmark computes the same objective terms used by ADMM and ADMM-MPC.

## 1. Mainline Delay

The mainline delay is calculated from the CTM state-based delay function:

$$
D_M(t)
=
\sum_{i=1}^{8}
D_{M,i}(t)
$$

Cell-level delay is clamped at zero so that artificial negative delay from draining cells cannot cancel real positive congestion delay:

$$
D_{M,i}(t)
=
\max\left(D^{raw}_{M,i}(t), 0\right)
$$

## 2. Local Ramp Delay

The local ramp delay is calculated from the average ramp queue during the timestep:

$$
D_L(t)
=
\sum_{j=1}^{5}
\frac{R_j(t) + R_j(t+1)}{2}
\cdot
\Delta t
$$

## 3. Fairness Penalty

The fairness penalty uses the capped ramp stress after the ramp queue update:

$$
\phi_j(t+1)
=
\min
\left(
\frac{R_j(t+1)}{R_{\max,j}},
1
\right)
$$

The fairness penalty is:

$$
L_{\text{fair}}(t)
=
\gamma
\sum_{i<j}
\left(
\phi_i(t+1) - \phi_j(t+1)
\right)^2
$$

## 4. Doorway Capacity Penalty

The doorway capacity penalty checks whether total inflow into a cell exceeds the cell doorway capacity:

$$
L_{\text{door}}(t)
=
\lambda_1
\sum_i
\left[
\max
\left(
q_{\text{in},i}(t) + u_{\text{in},i}(t) - C_i,
0
\right)
\right]^2
$$

## 5. Safe Threshold Penalty

The safe threshold penalty checks whether the next CTM state exceeds the desired safe occupancy threshold:

$$
L_{\text{safe}}(t)
=
\lambda_2
\sum_i
\left[
\max
\left(
x_i(t+1) - X_{\text{safe},i},
0
\right)
\right]^2
$$

## 6. Physical Capacity Penalty

The physical capacity penalty checks whether the next CTM state exceeds physical storage capacity:

$$
L_{\text{phys}}(t)
=
\lambda_3
\sum_i
\left[
\max
\left(
x_i(t+1) - N^{\max}_i,
0
\right)
\right]^2
$$

## 7. Spillback Penalty

The spillback penalty checks whether ramp demand exceeds ramp storage capacity:

$$
L_{\text{spill}}(t)
=
\lambda_4
\sum_j
\text{spillback}_j(t)^2
$$

## Total Capacity Penalty

The total capacity penalty at one timestep is:

$$
L_{\text{cap}}(t)
=
L_{\text{door}}(t)
+
L_{\text{safe}}(t)
+
L_{\text{phys}}(t)
+
L_{\text{spill}}(t)
$$

## Raw Benchmark Objective

The raw benchmark objective for each timestep is:

$$
J(t)
=
D_M(t)
+
D_L(t)
+
L_{\text{fair}}(t)
+
L_{\text{cap}}(t)
$$

---

## 17. State-Based CTM Benchmark Summary

After running all 120 steps, the benchmark totals are computed by summing each term over time.

The total mainline delay is:

$$
D_M
=
\sum_{t=0}^{119}
D_M(t)
$$

The total local ramp delay is:

$$
D_L
=
\sum_{t=0}^{119}
D_L(t)
$$

The total fairness penalty is:

$$
L_{\text{fair}}
=
\sum_{t=0}^{119}
L_{\text{fair}}(t)
$$

The total capacity penalty is:

$$
L_{\text{cap}}
=
\sum_{t=0}^{119}
L_{\text{cap}}(t)
$$

The final raw benchmark objective is:

$$
J_{\text{benchmark}}
=
\sum_{t=0}^{119}
J(t)
$$

This produces the official state-based field-observed ramp-release benchmark for comparison against ADMM and ADMM-MPC.

## Final Outputs

The summary table reports:

- total mainline delay
- total local ramp delay
- total fairness penalty
- doorway penalty
- safe-threshold penalty
- physical-capacity penalty
- spillback penalty
- total capacity penalty
- raw total objective

The notebook also reports:

- final mainline cell states after 120 steps
- final ramp queues after 120 steps
- history length checks to confirm that each tracked series contains 120 simulation steps

In [77]:
# 16. Official 120-step state-based CTM benchmark simulation
# Purpose: Run the field-observed ramp-release benchmark over the full 08:00–09:00 window.

def simulate_state_based_benchmark_120_steps(
    num_steps,
    mainline_initial_state,
    ramp_queue_0,
    q_in_boundary_series,
    observed_release_series,
    ramp_arrival_series,
    f_out_series,
    doorway_capacity,
    physical_capacity,
    safe_threshold_capacity,
    ramp_name_map,
    ramp_max_queue_named,
    ramp_max_queue_by_u,
    tt_ff_min,
    delta_t,
    gamma,
    lambda_1,
    lambda_2,
    lambda_3,
    lambda_4
):
    x_current = mainline_initial_state.copy()
    R_current = ramp_queue_0.copy()
    B_current = {
        ramp: 0.0
        for ramp in ramp_queue_0
    }

    history = {
        "step": [],
        "x": [],
        "R": [],
        "B": [],
        "q_in": [],
        "q_out": [],
        "u_in": [],
        "f_out": [],
        "actual_f_out": [],
        "observed_release": [],
        "actual_release": [],
        "ramp_arrival": [],
        "spillback": [],
        "mainline_delay": [],
        "local_delay": [],
        "fairness_penalty": [],
        "doorway_penalty": [],
        "safe_penalty": [],
        "physical_penalty": [],
        "spillback_penalty": [],
        "capacity_penalty": [],
        "total_objective": [],
    }

    for step in range(num_steps):

        # 1. Dynamic PeMS inputs for this 30-sec step
        q_in_boundary_step = q_in_boundary_series[step]
        observed_release_step = {
            "u1": observed_release_series["u1"][step],
            "u2": observed_release_series["u2"][step],
            "u3": observed_release_series["u3"][step],
            "u4": observed_release_series["u4"][step],
            "u5": observed_release_series["u5"][step],
        }

        ramp_arrival_step = {
            "u1": ramp_arrival_series["u1"][step],
            "u2": ramp_arrival_series["u2"][step],
            "u3": ramp_arrival_series["u3"][step],
            "u4": ramp_arrival_series["u4"][step],
            "u5": ramp_arrival_series["u5"][step],
        }

        f_out_step = f_out_series[step]

        # 2. Conservative ramp queue update with persistent external spillback
        (
            R_next,
            B_next,
            spillback_by_ramp,
            actual_release_step,
        ) = ramp_next_queue_with_spillback(
            R_current,
            B_current,
            ramp_arrival_step,
            observed_release_step,
            ramp_max_queue_by_u
        )

        # 3. Build freeway on-ramp inflow from actual releases, not commands.
        u_in_step = build_u_in_from_ramp_release(
            actual_release_step
        )

        # 4. Local ramp delay including external spillback/backlog
        _, total_local_delay = ramp_delay_with_cap(
            R_current,
            R_next,
            B_current,
            B_next,
            ramp_arrival_step,
            observed_release_step,
            actual_release_step,
            delta_t
        )

        # 5. Fairness penalty based on physical ramp storage stress
        _, _, L_fair = fairness_penalty_one_step(
            R_next,
            ramp_name_map,
            ramp_max_queue_named,
            gamma
        )

        # 6. CTM mainline state update
        x_next, q_out, q_in, sending, receiving, actual_f_out = ctm_30sec_step(
            x_current,
            q_in_boundary_step,
            u_in_step,
            f_out_step,
            doorway_capacity,
            physical_capacity
        )

        # 7. State-based mainline delay
        _, total_mainline_delay = mainline_delay_one_step(
            x_current,
            x_next,
            q_out,
            actual_f_out,
            tt_ff_min,
            delta_t
        )

        # 8. Capacity penalties
        capacity_info = capacity_penalty_one_step(
            q_in,
            u_in_step,
            x_next,
            doorway_capacity,
            safe_threshold_capacity,
            physical_capacity,
            spillback_by_ramp,
            lambda_1,
            lambda_2,
            lambda_3,
            lambda_4
        )

        # 9. Raw objective
        total_objective = (
            total_mainline_delay
            + total_local_delay
            + L_fair
            + capacity_info["total_capacity_penalty"]
        )

        # 10. Store results
        history["step"].append(step + 1)
        history["x"].append(x_next.copy())
        history["R"].append(R_next.copy())
        history["B"].append(B_next.copy())
        history["q_in"].append(q_in.copy())
        history["q_out"].append(q_out.copy())
        history["u_in"].append(u_in_step.copy())
        history["observed_release"].append(observed_release_step.copy())
        history["actual_release"].append(actual_release_step.copy())
        history["ramp_arrival"].append(ramp_arrival_step.copy())
        history["spillback"].append(spillback_by_ramp.copy())
        history["f_out"].append(f_out_step.copy())
        history["actual_f_out"].append(actual_f_out.copy())
        history["mainline_delay"].append(total_mainline_delay)
        history["local_delay"].append(total_local_delay)
        history["fairness_penalty"].append(L_fair)

        history["doorway_penalty"].append(
            capacity_info["total_doorway_penalty"]
        )

        history["safe_penalty"].append(
            capacity_info["total_safe_threshold_penalty"]
        )

        history["physical_penalty"].append(
            capacity_info["total_physical_capacity_penalty"]
        )

        history["spillback_penalty"].append(
            capacity_info["total_spillback_penalty"]
        )

        history["capacity_penalty"].append(
            capacity_info["total_capacity_penalty"]
        )

        history["total_objective"].append(total_objective)

        # 11. Move to next step
        x_current = x_next.copy()
        R_current = R_next.copy()
        B_current = B_next.copy()

    history["R_final"] = R_current.copy()
    history["B_final"] = B_current.copy()
    history["x_final"] = x_current.copy()

    return history


state_based_benchmark_history = simulate_state_based_benchmark_120_steps(
    num_steps=num_steps,
    mainline_initial_state=mainline_initial_state,
    ramp_queue_0=ramp_queue_0,
    q_in_boundary_series=q_in_boundary_series,
    observed_release_series=observed_release_series,
    ramp_arrival_series=ramp_arrival_series,
    f_out_series=f_out_series,
    doorway_capacity=doorway_capacity,
    physical_capacity=physical_capacity,
    safe_threshold_capacity=safe_threshold_capacity,
    ramp_name_map=ramp_name_map,
    ramp_max_queue_named=ramp_max_queue_named,
    ramp_max_queue_by_u=ramp_max_queue_by_u,
    tt_ff_min=tt_ff_min,
    delta_t=delta_t,
    gamma=gamma,
    lambda_1=lambda_1,
    lambda_2=lambda_2,
    lambda_3=lambda_3,
    lambda_4=lambda_4
)


In [90]:
# 16b. Ramp mass-conservation diagnostic
# Purpose: Verify that ramp arrivals are either released, stored on the ramp, or carried in the external spillback queue.

external_queue_0 = {
    ramp: 0.0
    for ramp in ramp_queue_0
}

total_initial_R = sum(
    float(ramp_queue_0[ramp])
    for ramp in ramp_queue_0
)

total_initial_B = sum(
    float(external_queue_0[ramp])
    for ramp in external_queue_0
)

total_arrivals = sum(
    float(ramp_arrival_series[ramp][step])
    for ramp in ramp_queue_0
    for step in range(num_steps)
)

total_actual_release = sum(
    float(state_based_benchmark_history["actual_release"][step][ramp])
    for ramp in ramp_queue_0
    for step in range(num_steps)
)

total_final_R = sum(
    float(state_based_benchmark_history["R_final"][ramp])
    for ramp in ramp_queue_0
)

total_final_B = sum(
    float(state_based_benchmark_history["B_final"][ramp])
    for ramp in ramp_queue_0
)

ramp_mass_residual = (
    total_initial_R
    + total_initial_B
    + total_arrivals
    - total_actual_release
    - total_final_R
    - total_final_B
)

ramp_mass_check_df = pd.DataFrame({
    "metric": [
        "initial_physical_ramp_queue",
        "initial_external_spillback_queue",
        "total_arrivals",
        "total_actual_release",
        "final_physical_ramp_queue",
        "final_external_spillback_queue",
        "mass_residual",
    ],
    "value": [
        total_initial_R,
        total_initial_B,
        total_arrivals,
        total_actual_release,
        total_final_R,
        total_final_B,
        ramp_mass_residual,
    ],
})

print("Ramp Mass-Conservation Check")
display(ramp_mass_check_df.round(8))

if abs(ramp_mass_residual) < 1e-8:
    print("PASS: ramp demand is conserved.")
else:
    print("FAIL: ramp demand is not conserved.")


Ramp Mass-Conservation Check


,metric,value
0,initial_physical_ramp_queue,0.0000
1,initial_external_spillback_queue,0.0000
2,total_arrivals,6300.0000
3,total_actual_release,4200.0000
4,final_physical_ramp_queue,278.4528
5,final_external_spillback_queue,1821.5472
6,mass_residual,0.0000


PASS: ramp demand is conserved.


In [91]:
# 17. State-based CTM benchmark summary
state_based_benchmark_summary_df = pd.DataFrame({
    "metric": [
        "Mainline Delay",
        "Local Ramp Delay",
        "Fairness Penalty",
        "Doorway Penalty",
        "Safe Threshold Penalty",
        "Physical Capacity Penalty",
        "Spillback Penalty",
        "Total Capacity Penalty",
        "Raw Total Objective",
    ],
    "value": [
        sum(state_based_benchmark_history["mainline_delay"]),
        sum(state_based_benchmark_history["local_delay"]),
        sum(state_based_benchmark_history["fairness_penalty"]),
        sum(state_based_benchmark_history["doorway_penalty"]),
        sum(state_based_benchmark_history["safe_penalty"]),
        sum(state_based_benchmark_history["physical_penalty"]),
        sum(state_based_benchmark_history["spillback_penalty"]),
        sum(state_based_benchmark_history["capacity_penalty"]),
        sum(state_based_benchmark_history["total_objective"]),
    ],
    "unit": [
        "veh-min",
        "veh-min",
        "penalty",
        "penalty",
        "penalty",
        "penalty",
        "penalty",
        "penalty",
        "raw objective",
    ],
})

print("State-Based CTM Benchmark: 8-cell / 30-sec / 08:00–09:00")
display(state_based_benchmark_summary_df.round(3))

# Final state
final_x = state_based_benchmark_history["x_final"]
final_R = state_based_benchmark_history["R_final"]
final_B = state_based_benchmark_history["B_final"]

final_x_df = pd.DataFrame({
    "cell": list(final_x.keys()),
    "final_x": list(final_x.values()),
    "safe_threshold_capacity": [
        safe_threshold_capacity[cell]
        for cell in final_x
    ],
    "physical_capacity": [
        physical_capacity[cell]
        for cell in final_x
    ],
})

final_R_df = pd.DataFrame({
    "ramp": list(final_R.keys()),
    "final_physical_queue_R": list(final_R.values()),
    "final_external_spillback_B": [
        final_B[ramp]
        for ramp in final_R
    ],
    "final_total_waiting_R_plus_B": [
        final_R[ramp] + final_B[ramp]
        for ramp in final_R
    ],
    "max_physical_queue": [
        ramp_max_queue_by_u[ramp]
        for ramp in final_R
    ],
})

print("Final Mainline State After 120 Steps")
display(final_x_df.round(3))

print("Final Ramp Queue and External Spillback After 120 Steps")
display(final_R_df.round(3))

# History length
history_length_check_df = pd.DataFrame({
    "history_series": [
        "mainline_delay",
        "local_delay",
        "fairness_penalty",
        "capacity_penalty",
        "total_objective",
        "x",
        "R",
        "B",
        "actual_release",
    ],
    "length": [
        len(state_based_benchmark_history["mainline_delay"]),
        len(state_based_benchmark_history["local_delay"]),
        len(state_based_benchmark_history["fairness_penalty"]),
        len(state_based_benchmark_history["capacity_penalty"]),
        len(state_based_benchmark_history["total_objective"]),
        len(state_based_benchmark_history["x"]),
        len(state_based_benchmark_history["R"]),
        len(state_based_benchmark_history["B"]),
        len(state_based_benchmark_history["actual_release"]),
    ],
})

print("History Length Check")
display(history_length_check_df)


State-Based CTM Benchmark: 8-cell / 30-sec / 08:00–09:00


,metric,value,unit
0,Mainline Delay,7.284869e+04,veh-min
1,Local Ramp Delay,6.348000e+04,veh-min
2,Fairness Penalty,2.545300e+01,penalty
3,Doorway Penalty,5.712108e+03,penalty
4,Safe Threshold Penalty,1.410495e+06,penalty
5,Physical Capacity Penalty,0.000000e+00,penalty
6,Spillback Penalty,1.334641e+07,penalty
7,Total Capacity Penalty,1.476262e+07,penalty
8,Raw Total Objective,1.489897e+07,raw objective


Final Mainline State After 120 Steps


,cell,final_x,safe_threshold_capacity,physical_capacity
0,Cell 1,66.200,413.744,591.062
1,Cell 2,73.300,413.744,591.062
2,Cell 3,104.791,413.744,591.062
3,Cell 4,520.336,413.744,591.062
4,Cell 5,633.148,496.492,709.275
5,Cell 6,341.895,413.744,591.062
6,Cell 7,630.948,496.492,709.275
7,Cell 8,159.833,413.744,591.062


Final Ramp Queue and External Spillback After 120 Steps


,ramp,final_physical_queue_R,final_external_spillback_B,final_total_waiting_R_plus_B,max_physical_queue
0,u1,28.669,475.331,504.0,28.669
1,u2,72.356,198.644,271.0,72.356
2,u3,56.168,446.332,502.5,56.168
3,u4,72.441,291.559,364.0,72.441
4,u5,48.819,409.681,458.5,48.819


History Length Check


,history_series,length
0,mainline_delay,120
1,local_delay,120
2,fairness_penalty,120
3,capacity_penalty,120
4,total_objective,120
5,x,120
6,R,120
7,B,120
8,actual_release,120


## Ramp Arrival Multiplier Sensitivity Check

Ramp arrival demand is not directly observed in the PeMS ramp detector data. The detector provides observed ramp release flow, but it does not directly measure the true number of vehicles arriving to the ramp queue. Therefore, the benchmark assumes ramp arrival demand as a multiple of the observed ramp release:

$$
a_j(t) = m \cdot u^{obs}_j(t)
$$

where:

- $a_j(t)$ = assumed arrival demand at ramp $j$ during time step $t$
- $u^{obs}_j(t)$ = observed ramp release flow at ramp $j$ during time step $t$
- $m$ = ramp arrival multiplier

The official benchmark uses:

$$
m = 1.5
$$

To check how sensitive the ramp-side results are to this assumption, four multipliers were tested: 1.0, 1.1, 1.25, and 1.5.

### Sensitivity Results

| Arrival Multiplier | Mainline Delay (veh-min) | Local Ramp Delay (veh-min) | Spillback Penalty |
|---:|---:|---:|---:|
| 1.00 | 71,909 | 0 | 0 |
| 1.10 | 71,909 | 9,785 | 64 |
| 1.25 | 71,909 | 13,799 | 747 |
| 1.50 | 71,909 | 15,226 | 3,454 |

### Interpretation

The mainline delay stays constant across all multiplier values because the benchmark uses the same observed ramp release series as the actual freeway inflow. Changing the arrival multiplier only changes the estimated ramp queues, local ramp delay, and spillback penalty.

When $m = 1.0$, assumed arrivals equal observed releases, so ramp queues do not grow and local ramp delay is zero. As the multiplier increases above 1.0, assumed demand exceeds observed discharge, causing ramp queues to grow. This increases local ramp delay and eventually creates spillback.

The official benchmark keeps $m = 1.5$ because the same arrival-demand assumption is used consistently across the benchmark, ADMM, and ADMM-MPC models. Therefore, the comparison between control methods remains fair. However, the absolute ramp-delay and spillback values should be interpreted as model-based estimates under the assumed demand scenario, not directly measured field delay.

### Final Modeling Choice

For the official benchmark, the ramp arrival demand is defined as:

$$
a_j(t) = 1.5 \cdot u^{obs}_j(t)
$$

Under this scenario, the sensitivity check produces:

- Mainline delay: **71,909 veh-min**
- Local ramp delay: **15,226 veh-min**
- Spillback penalty: **3,454**

This multiplier is kept fixed across the benchmark, ADMM, and ADMM-MPC experiments.

In [79]:
## arrival multiplayer sensitivity sweep

for mult in [1.0, 1.1, 1.25, 1.5]:
    arr = {
        r: [mult * v for v in observed_release_series[r]]
        for r in observed_release_series
    }

    hist = simulate_state_based_benchmark_120_steps(
        num_steps=num_steps,
        mainline_initial_state=mainline_initial_state,
        ramp_queue_0=ramp_queue_0,
        q_in_boundary_series=q_in_boundary_series,
        observed_release_series=observed_release_series,
        ramp_arrival_series=arr,
        f_out_series=f_out_series,
        doorway_capacity=doorway_capacity,
        physical_capacity=physical_capacity,
        safe_threshold_capacity=safe_threshold_capacity,
        ramp_name_map=ramp_name_map,
        ramp_max_queue_named=ramp_max_queue_named,
        ramp_max_queue_by_u=ramp_max_queue_by_u,
        tt_ff_min=tt_ff_min,
        delta_t=delta_t,
        gamma=gamma,
        lambda_1=lambda_1,
        lambda_2=lambda_2,
        lambda_3=lambda_3,
        lambda_4=lambda_4,
    )

    print(
        f"mult={mult}: "
        f"local={sum(hist['local_delay']):.0f}  "
        f"spillback={sum(hist['spillback_penalty']):.0f}  "
        f"mainline={sum(hist['mainline_delay']):.0f}"
    )

mult=1.0: local=0  spillback=0  mainline=72849
mult=1.1: local=12696  spillback=120466  mainline=72849
mult=1.25: local=31740  spillback=2304021  mainline=72849
mult=1.5: local=63480  spillback=13346411  mainline=72849


## 18–20. Full Benchmark History, Cumulative Totals, and Saved Benchmark Values

## Purpose

This section organizes the official state-based CTM benchmark results after the full 120-step simulation is complete.

It creates three outputs:

1. a full per-step benchmark history table
2. a cumulative benchmark summary table
3. saved benchmark totals for later comparison with ADMM and ADMM-MPC

---

## 18. Full 120-Step Benchmark History Summary

The full benchmark history table shows the objective components at every 30-second CTM timestep.

Each row corresponds to one CTM step:

$$
t = 1, 2, \dots, 120
$$

The table includes:

- mainline delay
- local ramp delay
- fairness penalty
- doorway penalty
- safe-threshold penalty
- physical-capacity penalty
- spillback penalty
- total capacity penalty
- total objective

The per-step objective is calculated as:

$$
J(t)
=
D_M(t)
+
D_L(t)
+
L_{\text{fair}}(t)
+
L_{\text{cap}}(t)
$$

where:

- $D_M(t)$ = mainline delay at step $t$
- $D_L(t)$ = local ramp delay at step $t$
- $L_{\text{fair}}(t)$ = fairness penalty at step $t$
- $L_{\text{cap}}(t)$ = total capacity penalty at step $t$

This table is useful for checking when congestion begins to accumulate and when different penalty terms become active.

For this benchmark, the safe-threshold penalty becomes active when any CTM cell state exceeds its safe occupancy threshold:

$$
x_i(t+1) > X_{\text{safe},i}
$$

The physical-capacity penalty remains zero as long as every cell stays below its physical storage capacity:

$$
x_i(t+1) \leq N^{\max}_i
$$

for all cells.

---

## 19. Cumulative 120-Step Benchmark Summary

The cumulative benchmark table shows how each objective component accumulates over time.

For example, cumulative mainline delay at step $k$ is:

$$
\text{CumD}_M(k)
=
\sum_{t=1}^{k}
D_M(t)
$$

Cumulative local ramp delay is:

$$
\text{CumD}_L(k)
=
\sum_{t=1}^{k}
D_L(t)
$$

Cumulative fairness penalty is:

$$
\text{CumL}_{\text{fair}}(k)
=
\sum_{t=1}^{k}
L_{\text{fair}}(t)
$$

Cumulative capacity penalty is:

$$
\text{CumL}_{\text{cap}}(k)
=
\sum_{t=1}^{k}
L_{\text{cap}}(t)
$$

The cumulative total objective is:

$$
\text{CumJ}(k)
=
\sum_{t=1}^{k}
J(t)
$$

At the final step:

$$
k = 120
$$

the cumulative totals must match the official benchmark summary.

Therefore, the final cumulative row is used as a consistency check for the benchmark calculation.

---

## 20. Save Official Benchmark Totals

After verifying the cumulative totals, the final benchmark values are saved into variables.

These saved totals are used later for direct comparison against:

- Greedy ADMM
- ADMM-MPC with static inputs
- ADMM-MPC with dynamic PeMS inputs

The saved official benchmark totals are:

- $D_M$ = total mainline delay
- $D_L$ = total local ramp delay
- $L_{\text{fair}}$ = total fairness penalty
- $L_{\text{door}}$ = total doorway penalty
- $L_{\text{safe}}$ = total safe-threshold penalty
- $L_{\text{phys}}$ = total physical-capacity penalty
- $L_{\text{spill}}$ = total spillback penalty
- $L_{\text{cap}}$ = total capacity penalty
- $J_{\text{benchmark}}$ = raw benchmark objective

The final saved benchmark objective is:

$$
J_{\text{benchmark}}
=
D_M
+
D_L
+
L_{\text{fair}}
+
L_{\text{cap}}
$$

where:

$$
L_{\text{cap}}
=
L_{\text{door}}
+
L_{\text{safe}}
+
L_{\text{phys}}
+
L_{\text{spill}}
$$

These saved values complete the official state-based CTM benchmark section and provide the field-observed ramp-release baseline for later ADMM and ADMM-MPC comparisons.

In [80]:
# 18. Full 120-step benchmark history summary
# Purpose: Show every 30-second CTM step from 08:00 to 09:00.

benchmark_step_summary_df = pd.DataFrame({
    "step": state_based_benchmark_history["step"],
    "mainline_delay": state_based_benchmark_history["mainline_delay"],
    "local_delay": state_based_benchmark_history["local_delay"],
    "fairness_penalty": state_based_benchmark_history["fairness_penalty"],
    "doorway_penalty": state_based_benchmark_history["doorway_penalty"],
    "safe_penalty": state_based_benchmark_history["safe_penalty"],
    "physical_penalty": state_based_benchmark_history["physical_penalty"],
    "spillback_penalty": state_based_benchmark_history["spillback_penalty"],
    "capacity_penalty": state_based_benchmark_history["capacity_penalty"],
    "total_objective": state_based_benchmark_history["total_objective"],
    "total_physical_ramp_queue_R": [
        sum(step_R.values())
        for step_R in state_based_benchmark_history["R"]
    ],
    "total_external_spillback_B": [
        sum(step_B.values())
        for step_B in state_based_benchmark_history["B"]
    ],
    "total_actual_release": [
        sum(step_release.values())
        for step_release in state_based_benchmark_history["actual_release"]
    ],
})

print("Full 120-Step Benchmark Summary")
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

display(benchmark_step_summary_df.round(3))


Full 120-Step Benchmark Summary


,step,mainline_delay,local_delay,fairness_penalty,doorway_penalty,safe_penalty,physical_penalty,spillback_penalty,capacity_penalty,total_objective
0,1,209.268,4.000,0.049,124.010,0.000,0.0,0.000,124.010,337.327
1,2,212.763,12.000,0.197,124.010,0.000,0.0,0.000,124.010,348.970
2,3,223.672,20.000,0.443,87.990,0.000,0.0,0.000,87.990,332.104
3,4,229.624,28.000,0.788,46.570,0.000,0.0,0.000,46.570,304.982
4,5,231.211,36.000,1.231,46.570,0.000,0.0,0.000,46.570,315.012
5,6,232.797,44.000,1.773,46.570,0.000,0.0,0.000,46.570,325.140
6,7,234.384,52.000,2.000,46.570,0.000,0.0,2.270,48.840,337.224
7,8,235.971,60.000,1.837,46.570,0.000,0.0,21.326,67.896,365.703
8,9,241.338,68.000,1.703,46.570,0.000,0.0,59.741,106.311,417.352
9,10,247.875,76.000,1.600,46.570,0.000,0.0,117.517,164.087,489.562


In [81]:
# 19. Cumulative 120-step benchmark summary
# Purpose: Show how benchmark delay and penalties accumulate over time. The final row should match the total benchmark summary.

benchmark_cumulative_summary_df = benchmark_step_summary_df.copy()

benchmark_cumulative_summary_df["cum_mainline_delay"] = (
    benchmark_cumulative_summary_df["mainline_delay"].cumsum()
)

benchmark_cumulative_summary_df["cum_local_delay"] = (
    benchmark_cumulative_summary_df["local_delay"].cumsum()
)

benchmark_cumulative_summary_df["cum_fairness_penalty"] = (
    benchmark_cumulative_summary_df["fairness_penalty"].cumsum()
)

benchmark_cumulative_summary_df["cum_doorway_penalty"] = (
    benchmark_cumulative_summary_df["doorway_penalty"].cumsum()
)

benchmark_cumulative_summary_df["cum_safe_penalty"] = (
    benchmark_cumulative_summary_df["safe_penalty"].cumsum()
)

benchmark_cumulative_summary_df["cum_physical_penalty"] = (
    benchmark_cumulative_summary_df["physical_penalty"].cumsum()
)

benchmark_cumulative_summary_df["cum_spillback_penalty"] = (
    benchmark_cumulative_summary_df["spillback_penalty"].cumsum()
)

benchmark_cumulative_summary_df["cum_capacity_penalty"] = (
    benchmark_cumulative_summary_df["capacity_penalty"].cumsum()
)

benchmark_cumulative_summary_df["cum_total_objective"] = (
    benchmark_cumulative_summary_df["total_objective"].cumsum()
)


print("Cumulative 120-Step Benchmark Summary")
display(benchmark_cumulative_summary_df.round(3))


print("Final cumulative totals:")
display(
    benchmark_cumulative_summary_df.tail(1)[[
        "cum_mainline_delay",
        "cum_local_delay",
        "cum_fairness_penalty",
        "cum_doorway_penalty",
        "cum_safe_penalty",
        "cum_physical_penalty",
        "cum_spillback_penalty",
        "cum_capacity_penalty",
        "cum_total_objective",
    ]].round(3)
)

Cumulative 120-Step Benchmark Summary


,step,mainline_delay,local_delay,fairness_penalty,doorway_penalty,safe_penalty,physical_penalty,spillback_penalty,capacity_penalty,total_objective,cum_mainline_delay,cum_local_delay,cum_fairness_penalty,cum_doorway_penalty,cum_safe_penalty,cum_physical_penalty,cum_spillback_penalty,cum_capacity_penalty,cum_total_objective
0,1,209.268,4.000,0.049,124.010,0.000,0.0,0.000,124.010,337.327,209.268,4.000,0.049,124.010,0.000,0.0,0.000000e+00,1.240100e+02,3.373270e+02
1,2,212.763,12.000,0.197,124.010,0.000,0.0,0.000,124.010,348.970,422.030,16.000,0.246,248.020,0.000,0.0,0.000000e+00,2.480200e+02,6.862960e+02
2,3,223.672,20.000,0.443,87.990,0.000,0.0,0.000,87.990,332.104,645.702,36.000,0.689,336.010,0.000,0.0,0.000000e+00,3.360100e+02,1.018401e+03
3,4,229.624,28.000,0.788,46.570,0.000,0.0,0.000,46.570,304.982,875.326,64.000,1.477,382.580,0.000,0.0,0.000000e+00,3.825800e+02,1.323383e+03
4,5,231.211,36.000,1.231,46.570,0.000,0.0,0.000,46.570,315.012,1106.536,100.000,2.708,429.150,0.000,0.0,0.000000e+00,4.291500e+02,1.638394e+03
5,6,232.797,44.000,1.773,46.570,0.000,0.0,0.000,46.570,325.140,1339.334,144.000,4.481,475.720,0.000,0.0,0.000000e+00,4.757200e+02,1.963534e+03
6,7,234.384,52.000,2.000,46.570,0.000,0.0,2.270,48.840,337.224,1573.717,196.000,6.482,522.290,0.000,0.0,2.270000e+00,5.245600e+02,2.300759e+03
7,8,235.971,60.000,1.837,46.570,0.000,0.0,21.326,67.896,365.703,1809.688,256.000,8.318,568.860,0.000,0.0,2.359600e+01,5.924550e+02,2.666462e+03
8,9,241.338,68.000,1.703,46.570,0.000,0.0,59.741,106.311,417.352,2051.026,324.000,10.021,615.430,0.000,0.0,8.333700e+01,6.987670e+02,3.083814e+03
9,10,247.875,76.000,1.600,46.570,0.000,0.0,117.517,164.087,489.562,2298.901,400.000,11.622,662.000,0.000,0.0,2.008540e+02,8.628530e+02,3.573376e+03


Final cumulative totals:


,cum_mainline_delay,cum_local_delay,cum_fairness_penalty,cum_doorway_penalty,cum_safe_penalty,cum_physical_penalty,cum_spillback_penalty,cum_capacity_penalty,cum_total_objective
119,72848.691,63480.0,25.453,5712.108,1410495.372,0.0,1.334641e+07,1.476262e+07,14898973.08


In [82]:

# 20. Save official benchmark totals
# Purpose: Store final benchmark totals for later comparison with ADMM and ADMM-MPC.

benchmark_mainline_delay = sum(state_based_benchmark_history["mainline_delay"])
benchmark_local_delay = sum(state_based_benchmark_history["local_delay"])
benchmark_fairness_penalty = sum(state_based_benchmark_history["fairness_penalty"])
benchmark_doorway_penalty = sum(state_based_benchmark_history["doorway_penalty"])
benchmark_safe_penalty = sum(state_based_benchmark_history["safe_penalty"])
benchmark_physical_penalty = sum(state_based_benchmark_history["physical_penalty"])
benchmark_spillback_penalty = sum(state_based_benchmark_history["spillback_penalty"])
benchmark_capacity_penalty = sum(state_based_benchmark_history["capacity_penalty"])
benchmark_raw_objective = sum(state_based_benchmark_history["total_objective"])

benchmark_service_metrics = {
    "total_ramp_arrivals": total_arrivals,
    "total_actual_release": total_actual_release,
    "final_physical_ramp_queue_R": total_final_R,
    "final_external_spillback_queue_B": total_final_B,
    "ramp_mass_residual": ramp_mass_residual,
}

total_ramp_demand_to_account = (
    total_initial_R
    + total_initial_B
    + total_arrivals
)

benchmark_service_metrics["total_ramp_demand_to_account"] = total_ramp_demand_to_account

if total_ramp_demand_to_account > 0:
    benchmark_service_metrics["served_fraction"] = (
        benchmark_service_metrics["total_actual_release"]
        / total_ramp_demand_to_account
    )
else:
    benchmark_service_metrics["served_fraction"] = 1.0
official_benchmark_totals = {
    "mainline_delay": benchmark_mainline_delay,
    "local_delay": benchmark_local_delay,
    "fairness_penalty": benchmark_fairness_penalty,
    "doorway_penalty": benchmark_doorway_penalty,
    "safe_penalty": benchmark_safe_penalty,
    "physical_penalty": benchmark_physical_penalty,
    "spillback_penalty": benchmark_spillback_penalty,
    "capacity_penalty": benchmark_capacity_penalty,
    "raw_objective": benchmark_raw_objective,
}

official_benchmark_totals_df = pd.DataFrame({
    "metric": list(official_benchmark_totals.keys()),
    "value": list(official_benchmark_totals.values()),
})

benchmark_service_metrics_df = pd.DataFrame({
    "metric": list(benchmark_service_metrics.keys()),
    "value": list(benchmark_service_metrics.values()),
})

print("Official State-Based Benchmark Totals")
display(official_benchmark_totals_df.round(3))

print("Benchmark Ramp Service / Conservation Metrics")
display(benchmark_service_metrics_df.round(6))


Official State-Based Benchmark Totals


,metric,value
0,mainline_delay,7.284869e+04
1,local_delay,6.348000e+04
2,fairness_penalty,2.545300e+01
3,doorway_penalty,5.712108e+03
4,safe_penalty,1.410495e+06
5,physical_penalty,0.000000e+00
6,spillback_penalty,1.334641e+07
7,capacity_penalty,1.476262e+07
8,raw_objective,1.489897e+07


In [83]:
#(changed)
# EXPORT SOURCE-OF-TRUTH BENCHMARK INPUTS FOR ADMM-MPC
# Purpose: Re-run the official corrected benchmark and export the exact inputs, totals, and normalization denominators used by ADMM-MPC.

import pickle
import pandas as pd

# 1. Official ramp/cell mapping

ramp_ids = ["u1", "u2", "u3", "u4", "u5"]

ramp_cell_map = {
    "u1": "Cell 2",   # Bristol 1 OR
    "u2": "Cell 4",   # Fairview OR
    "u3": "Cell 5",   # Harbor 1 OR
    "u4": "Cell 6",   # Harbor 2 OR
    "u5": "Cell 7",   # Euclid OR
}

external_queue_0 = {
    ramp: 0.0
    for ramp in ramp_ids
}

# 2. Re-run official benchmark from the current corrected inputs

official_benchmark_history = simulate_state_based_benchmark_120_steps(
    num_steps=num_steps,
    mainline_initial_state=mainline_initial_state,
    ramp_queue_0=ramp_queue_0,
    q_in_boundary_series=q_in_boundary_series,
    observed_release_series=observed_release_series,
    ramp_arrival_series=ramp_arrival_series,
    f_out_series=f_out_series,
    doorway_capacity=doorway_capacity,
    physical_capacity=physical_capacity,
    safe_threshold_capacity=safe_threshold_capacity,
    ramp_name_map=ramp_name_map,
    ramp_max_queue_named=ramp_max_queue_named,
    ramp_max_queue_by_u=ramp_max_queue_by_u,
    tt_ff_min=tt_ff_min,
    delta_t=delta_t,
    gamma=gamma,
    lambda_1=lambda_1,
    lambda_2=lambda_2,
    lambda_3=lambda_3,
    lambda_4=lambda_4,
)

# 3. Summarize official benchmark totals

official_totals = {
    "mainline_delay": sum(official_benchmark_history["mainline_delay"]),
    "local_delay": sum(official_benchmark_history["local_delay"]),
    "fairness_penalty": sum(official_benchmark_history["fairness_penalty"]),
    "doorway_penalty": sum(official_benchmark_history["doorway_penalty"]),
    "safe_penalty": sum(official_benchmark_history["safe_penalty"]),
    "physical_penalty": sum(official_benchmark_history["physical_penalty"]),
    "spillback_penalty": sum(official_benchmark_history["spillback_penalty"]),
}

official_totals["capacity_penalty"] = (
    official_totals["doorway_penalty"]
    + official_totals["safe_penalty"]
    + official_totals["physical_penalty"]
    + official_totals["spillback_penalty"]
)

official_totals["raw_objective"] = (
    official_totals["mainline_delay"]
    + official_totals["local_delay"]
    + official_totals["fairness_penalty"]
    + official_totals["capacity_penalty"]
)

official_totals_df = pd.DataFrame(
    official_totals.items(),
    columns=["term", "value"]
)

print("Official benchmark totals exported to shared_benchmark_inputs.pkl")
display(official_totals_df.round(6))

# 3b. Ramp service/conservation metrics for downstream diagnostics

official_service_metrics = {
    "total_ramp_arrivals": sum(
        float(ramp_arrival_series[ramp][step])
        for ramp in ramp_ids
        for step in range(num_steps)
    ),
    "total_actual_release": sum(
        float(official_benchmark_history["actual_release"][step][ramp])
        for ramp in ramp_ids
        for step in range(num_steps)
    ),
    "final_physical_ramp_queue_R": sum(
        float(official_benchmark_history["R_final"][ramp])
        for ramp in ramp_ids
    ),
    "final_external_spillback_queue_B": sum(
        float(official_benchmark_history["B_final"][ramp])
        for ramp in ramp_ids
    ),
}

official_service_metrics["ramp_mass_residual"] = (
    sum(float(ramp_queue_0[ramp]) for ramp in ramp_ids)
    + sum(float(external_queue_0[ramp]) for ramp in ramp_ids)
    + official_service_metrics["total_ramp_arrivals"]
    - official_service_metrics["total_actual_release"]
    - official_service_metrics["final_physical_ramp_queue_R"]
    - official_service_metrics["final_external_spillback_queue_B"]
)

official_total_ramp_demand_to_account = (
    sum(float(ramp_queue_0[ramp]) for ramp in ramp_ids)
    + sum(float(external_queue_0[ramp]) for ramp in ramp_ids)
    + official_service_metrics["total_ramp_arrivals"]
)

official_service_metrics["total_ramp_demand_to_account"] = (
    official_total_ramp_demand_to_account
)

if official_total_ramp_demand_to_account > 0:
    official_service_metrics["served_fraction"] = (
        official_service_metrics["total_actual_release"]
        / official_total_ramp_demand_to_account
    )
else:
    official_service_metrics["served_fraction"] = 1.0

print("Official benchmark ramp service / conservation metrics")
display(pd.DataFrame(official_service_metrics.items(), columns=["metric", "value"]).round(6))

# 4. Compute normalization denominators

def safe_divide_or_one(numerator, denominator):
    if denominator == 0:
        return 1.0

    value = numerator / denominator

    if value == 0:
        return 1.0

    return value

benchmark_denominators = {
    "D_main_base": official_totals["mainline_delay"] or 1.0,
    "D_local_base": official_totals["local_delay"] or 1.0,

    "L_fair_base": safe_divide_or_one(
        official_totals["fairness_penalty"],
        gamma,
    ),

    "P_door_base": safe_divide_or_one(
        official_totals["doorway_penalty"],
        lambda_1,
    ),

    "P_safe_base": safe_divide_or_one(
        official_totals["safe_penalty"],
        lambda_2,
    ),

    "P_phys_base": safe_divide_or_one(
        official_totals["physical_penalty"],
        lambda_3,
    ),

    "P_spill_base": safe_divide_or_one(
        official_totals["spillback_penalty"],
        lambda_4,
    ),
}

benchmark_denominators_df = pd.DataFrame(
    benchmark_denominators.items(),
    columns=["denominator", "value"]
)

print("Benchmark normalization denominators")
display(benchmark_denominators_df.round(6))

# 5. Build shared source-of-truth dictionary

shared_benchmark_inputs = {
    # core simulation settings
    "num_steps": num_steps,
    "delta_t": delta_t,
    "arrival_multiplier": arrival_multiplier,

    # objective weights / penalty coefficients
    "gamma": gamma,
    "lambda_1": lambda_1,
    "lambda_2": lambda_2,
    "lambda_3": lambda_3,
    "lambda_4": lambda_4,

    # CTM initial states
    "mainline_initial_state": mainline_initial_state,
    "ramp_queue_0": ramp_queue_0,
    "external_queue_0": external_queue_0,

    # benchmark time series
    "q_in_boundary_series": q_in_boundary_series,
    "observed_release_series": observed_release_series,
    "ramp_arrival_series": ramp_arrival_series,
    "f_out_series": f_out_series,

    # capacities
    "doorway_capacity": doorway_capacity,
    "physical_capacity": physical_capacity,
    "safe_threshold_capacity": safe_threshold_capacity,

    # ramp metadata
    "ramp_ids": ramp_ids,
    "ramp_cell_map": ramp_cell_map,
    "ramp_name_map": ramp_name_map,
    "ramp_max_queue_by_u": ramp_max_queue_by_u,
    "ramp_max_queue_named": ramp_max_queue_named,

    # free-flow travel time
    "tt_ff_min": tt_ff_min,

    # official benchmark results
    "official_benchmark_history": official_benchmark_history,
    "official_totals": official_totals,
    "official_service_metrics": official_service_metrics,

    # normalization denominators
    "benchmark_denominators": benchmark_denominators,
}

# 6. Save pickle file

with open("shared_benchmark_inputs.pkl", "wb") as fh:
    pickle.dump(shared_benchmark_inputs, fh)

print("Saved shared_benchmark_inputs.pkl")

# 7. Reload and verify export consistency

with open("shared_benchmark_inputs.pkl", "rb") as fh:
    shared_check = pickle.load(fh)

max_total_error = max(
    abs(
        float(shared_check["official_totals"][key])
        - float(official_totals[key])
    )
    for key in official_totals
)

max_denominator_error = max(
    abs(
        float(shared_check["benchmark_denominators"][key])
        - float(benchmark_denominators[key])
    )
    for key in benchmark_denominators
)

max_service_error = max(
    abs(
        float(shared_check["official_service_metrics"][key])
        - float(official_service_metrics[key])
    )
    for key in official_service_metrics
)

print("Max official total export mismatch:", max_total_error)
print("Max denominator export mismatch:", max_denominator_error)
print("Max service metric export mismatch:", max_service_error)

if max_total_error < 1e-9 and max_denominator_error < 1e-9 and max_service_error < 1e-9:
    print("PASS: shared_benchmark_inputs.pkl was freshly exported from the current benchmark.")
else:
    print("FAIL: shared_benchmark_inputs.pkl does not match current benchmark values.")


Official benchmark totals exported to shared_benchmark_inputs.pkl


,term,value
0,mainline_delay,7.284869e+04
1,local_delay,6.348000e+04
2,fairness_penalty,2.545297e+01
3,doorway_penalty,5.712108e+03
4,safe_penalty,1.410495e+06
5,physical_penalty,0.000000e+00
6,spillback_penalty,1.334641e+07
7,capacity_penalty,1.476262e+07
8,raw_objective,1.489897e+07


Benchmark normalization denominators


,denominator,value
0,D_main_base,7.284869e+04
1,D_local_base,6.348000e+04
2,L_fair_base,2.545297e+01
3,P_door_base,5.712108e+03
4,P_safe_base,2.820991e+06
5,P_phys_base,1.000000e+00
6,P_spill_base,2.669282e+07


Saved shared_benchmark_inputs.pkl
Max official total export mismatch: 0.0
Max denominator export mismatch: 0.0
PASS: shared_benchmark_inputs.pkl was freshly exported from the current benchmark.


# final diagnostic check

In [84]:

final_x = state_based_benchmark_history["x"][-1]

print("Final mainline state:")
for cell, value in final_x.items():
    print(
        cell,
        "x =", round(value, 3),
        "safe =", round(safe_threshold_capacity[cell], 3),
        "physical =", round(physical_capacity[cell], 3),
        "overflow_safe =", round(max(value - safe_threshold_capacity[cell], 0.0), 3)
    )

Final mainline state:
Cell 1 x = 66.2 safe = 413.744 physical = 591.062 overflow_safe = 0.0
Cell 2 x = 73.3 safe = 413.744 physical = 591.062 overflow_safe = 0.0
Cell 3 x = 104.791 safe = 413.744 physical = 591.062 overflow_safe = 0.0
Cell 4 x = 520.336 safe = 413.744 physical = 591.062 overflow_safe = 106.592
Cell 5 x = 633.148 safe = 496.493 physical = 709.275 overflow_safe = 136.656
Cell 6 x = 341.895 safe = 413.744 physical = 591.062 overflow_safe = 0.0
Cell 7 x = 630.948 safe = 496.493 physical = 709.275 overflow_safe = 134.456
Cell 8 x = 159.833 safe = 413.744 physical = 591.062 overflow_safe = 0.0


In [85]:
safe_by_cell = {f"Cell {i}": 0.0 for i in range(1, 9)}

x_hist = state_based_benchmark_history["x"]

if len(x_hist) == num_steps + 1:
    x_steps = x_hist[1:]
else:
    x_steps = x_hist[:num_steps]

for x_step in x_steps:
    for cell in safe_by_cell:
        overflow = max(x_step[cell] - safe_threshold_capacity[cell], 0.0)
        safe_by_cell[cell] += lambda_2 * overflow ** 2

print("Safe-threshold penalty by cell:")
for cell, value in safe_by_cell.items():
    print(cell, round(value, 3))

print("\nTotal recomputed safe penalty:")
print(round(sum(safe_by_cell.values()), 3))

print("\nReported safe penalty:")
print(round(sum(state_based_benchmark_history["safe_penalty"]), 3))

Safe-threshold penalty by cell:
Cell 1 0.0
Cell 2 0.0
Cell 3 0.0
Cell 4 203488.593
Cell 5 672755.402
Cell 6 0.0
Cell 7 534251.377
Cell 8 0.0

Total recomputed safe penalty:
1410495.372

Reported safe penalty:
1410495.372


## Doorway Capacity Source Tracking and Sensitivity Check

The CTM doorway capacity represents the maximum number of vehicles that can pass through each mainline cell during one 30-second timestep. For most stations, doorway capacity is directly available from the station-level capacity data. However, RED HILL and FAIRVIEW do not have directly given doorway capacities.

For these missing stations, doorway capacity is estimated using the average per-lane doorway capacity from stations with available capacity values. The estimated station capacity is computed as:

$$[
C_s = \bar{C}_{\text{per-lane}} \cdot N_s
]$$

where:

$$[
\bar{C}_{\text{per-lane}}
]$$

is the average per-lane doorway capacity from stations with known capacity values, and \(N_s\) is the number of lanes at the station with missing capacity.

To make this assumption explicit, the notebook records whether each station doorway capacity is directly given or estimated. A sensitivity check is then performed by scaling only the estimated doorway capacities for RED HILL and FAIRVIEW by:

$$[
0.90,\quad 1.00,\quad 1.10
]$$

while keeping all directly given doorway capacities fixed.

The sensitivity results show that this assumption materially affects the benchmark objective. Reducing the estimated capacities by 10% increases the raw objective by approximately 9.6%, while increasing the estimated capacities by 10% decreases the raw objective by approximately 21.1%. The largest effect appears in the safe-threshold penalty.

Therefore, doorway-capacity estimation is treated as an explicit modeling assumption rather than a negligible preprocessing detail.

In [86]:
# Doorway capacity source tracking
# Purpose: Track which doorway capacities are directly given and which are estimated from average per-lane capacity.

doorway_capacity_source = {}

for station in doorway_capacity_given:
    doorway_capacity_source[station] = "given"

for station in estimated_capacity:
    doorway_capacity_source[station] = "estimated_avg_per_lane"


# Station lane counts used for doorway-capacity source table.
# These are station-level lanes, not CTM cell lanes.
doorway_station_lanes = {
    "Bristol 1": 5,
    "Harbor 1": 6,
    "Harbor 2": 5,
    "Euclid": 6,
    "Talbert": 5,
    "RED HILL": 5,
    "FAIRVIEW": 5,
}


# Safety check: every station in doorway_capacity_station_5min
# must have a lane count.
missing_lane_keys = [
    station
    for station in doorway_capacity_station_5min
    if station not in doorway_station_lanes
]

if len(missing_lane_keys) > 0:
    raise KeyError(
        "Missing doorway station lane counts for: "
        + str(missing_lane_keys)
    )


doorway_station_capacity_source_df = pd.DataFrame({
    "station": list(doorway_capacity_station_5min.keys()),
    "capacity_source": [
        doorway_capacity_source[station]
        for station in doorway_capacity_station_5min
    ],
    "lanes": [
        doorway_station_lanes[station]
        for station in doorway_capacity_station_5min
    ],
    "doorway_capacity_veh_5min": [
        doorway_capacity_station_5min[station]
        for station in doorway_capacity_station_5min
    ],
    "per_lane_capacity_veh_5min": [
        doorway_capacity_station_5min[station] / doorway_station_lanes[station]
        for station in doorway_capacity_station_5min
    ],
})

print("Doorway capacity source table")
display(doorway_station_capacity_source_df.round(3))

Doorway capacity source table


,station,capacity_source,lanes,doorway_capacity_veh_5min,per_lane_capacity_veh_5min
0,Bristol 1,given,5,895.000,179.000
1,Harbor 1,given,6,1029.000,171.500
2,Harbor 2,given,5,860.000,172.000
3,Euclid,given,6,925.000,154.167
4,Talbert,given,5,833.000,166.600
5,RED HILL,estimated_avg_per_lane,5,843.267,168.653
6,FAIRVIEW,estimated_avg_per_lane,5,843.267,168.653


In [87]:
#  Doorway capacity sensitivity check
# Purpose: Test whether benchmark totals are sensitive to estimated doorway capacities for RED HILL and FAIRVIEW.

doorway_lane_lookup = dict(
    zip(
        doorway_station_capacity_source_df["station"],
        doorway_station_capacity_source_df["lanes"],
    )
)
def build_doorway_capacity_with_estimate_scale(scale):

    adjusted_station_capacity_5min = {}

    for station in doorway_capacity_station_5min:
        if doorway_capacity_source[station] == "estimated_avg_per_lane":
            adjusted_station_capacity_5min[station] = (
                doorway_capacity_station_5min[station] * scale
            )
        else:
            adjusted_station_capacity_5min[station] = (
                doorway_capacity_station_5min[station]
            )

    adjusted_per_lane_values = {
        station: (
            adjusted_station_capacity_5min[station]
            / doorway_lane_lookup[station]
        )
        for station in adjusted_station_capacity_5min
    }

    adjusted_avg_per_lane_5min = (
        sum(adjusted_per_lane_values.values())
        / len(adjusted_per_lane_values)
    )

    adjusted_per_lane_30sec = (adjusted_avg_per_lane_5min / 5.0) * delta_t

    adjusted_doorway_capacity = {
        f"Cell {i}": adjusted_per_lane_30sec * cell_lane_count[i]
        for i in range(1, 9)
    }

    return adjusted_doorway_capacity


doorway_sensitivity_rows = []

for scale in [0.90, 1.00, 1.10]:

    doorway_capacity_test = build_doorway_capacity_with_estimate_scale(scale)

    history_test = simulate_state_based_benchmark_120_steps(
        mainline_initial_state=mainline_initial_state,
        ramp_queue_0=ramp_queue_0,
        q_in_boundary_series=q_in_boundary_series,
        observed_release_series=observed_release_series,
        ramp_arrival_series=ramp_arrival_series,
        f_out_series=f_out_series,
        doorway_capacity=doorway_capacity_test,
        physical_capacity=physical_capacity,
        safe_threshold_capacity=safe_threshold_capacity,
        ramp_max_queue_by_u=ramp_max_queue_by_u,
        ramp_name_map=ramp_name_map,
        ramp_max_queue_named=ramp_max_queue_named,
        tt_ff_min=tt_ff_min,
        delta_t=delta_t,
        gamma=gamma,
        lambda_1=lambda_1,
        lambda_2=lambda_2,
        lambda_3=lambda_3,
        lambda_4=lambda_4,
        num_steps=num_steps,
        )

    mainline_delay_test = sum(history_test["mainline_delay"])
    local_delay_test = sum(history_test["local_delay"])
    fairness_penalty_test = sum(history_test["fairness_penalty"])
    doorway_penalty_test = sum(history_test["doorway_penalty"])
    safe_penalty_test = sum(history_test["safe_penalty"])
    physical_penalty_test = sum(history_test["physical_penalty"])
    spillback_penalty_test = sum(history_test["spillback_penalty"])
    capacity_penalty_test = sum(history_test["capacity_penalty"])

    raw_objective_test = (
        mainline_delay_test
        + local_delay_test
        + fairness_penalty_test
        + capacity_penalty_test
    )

    doorway_sensitivity_rows.append({
        "estimated_capacity_scale": scale,
        "mainline_delay": mainline_delay_test,
        "local_delay": local_delay_test,
        "fairness_penalty": fairness_penalty_test,
        "doorway_penalty": doorway_penalty_test,
        "safe_penalty": safe_penalty_test,
        "physical_penalty": physical_penalty_test,
        "spillback_penalty": spillback_penalty_test,
        "capacity_penalty": capacity_penalty_test,
        "raw_objective": raw_objective_test,
    })


doorway_capacity_sensitivity_df = pd.DataFrame(doorway_sensitivity_rows)

baseline_raw_objective = doorway_capacity_sensitivity_df.loc[
    doorway_capacity_sensitivity_df["estimated_capacity_scale"] == 1.00,
    "raw_objective"
].iloc[0]

doorway_capacity_sensitivity_df["raw_objective_change_pct"] = (
    100.0
    * (
        doorway_capacity_sensitivity_df["raw_objective"]
        - baseline_raw_objective
    )
    / baseline_raw_objective
)

print("Doorway capacity sensitivity check")
display(doorway_capacity_sensitivity_df.round(3))

Doorway capacity sensitivity check


,estimated_capacity_scale,mainline_delay,local_delay,fairness_penalty,doorway_penalty,safe_penalty,physical_penalty,spillback_penalty,capacity_penalty,raw_objective,raw_objective_change_pct
0,0.9,82109.323,63480.0,25.453,6252.757,1544918.512,0.0,1.334641e+07,1.489758e+07,1.504320e+07,0.968
1,1.0,72848.691,63480.0,25.453,5712.108,1410495.372,0.0,1.334641e+07,1.476262e+07,1.489897e+07,0.000
2,1.1,63631.078,63480.0,25.453,5170.547,1102660.122,0.0,1.334641e+07,1.445424e+07,1.458138e+07,-2.132


## Conservative Off-Ramp Outflow Check

The CTM update was modified so that requested off-ramp outflow cannot remove more vehicles than are physically available in a cell.

Previously, the state update used:

$$[
x_i(t+1) = x_i(t) + q_i^{in}(t) + u_i(t) - q_i^{out}(t) - f_i^{out}(t)
]$$

followed by nonnegative clipping:

$$[
x_i(t+1) = \max(x_i(t+1), 0)
]$$

This can hide edge cases where the requested off-ramp outflow exceeds the vehicles available in the cell. To make the CTM update conservative, the actual off-ramp outflow is capped by the vehicles remaining after mainline outflow:

$$[
\tilde{f}_i^{out}(t)
=
\min\left(
f_i^{out}(t),
\max\left[
0,\,
x_i(t) + q_i^{in}(t) + u_i(t) - q_i^{out}(t)
\right]
\right)
]$$

The CTM state update then uses $(\tilde{f}_i^{out}(t))$ instead of the requested $(f_i^{out}(t))$.

The diagnostic below checks whether any off-ramp outflow had to be capped during the benchmark simulation. In the official 08:00–09:00 benchmark, no capped off-ramp events occurred, meaning the correction improves edge-case physical consistency without changing the benchmark trajectory.

### Soft-Capacity Diagnostic Interpretation



The soft-capacity diagnostic shows that the benchmark trajectory contains capacity-pressure events. These events do not mean the CTM simulation failed. They indicate that the model allows excess combined inflow or high cell storage to occur, then records the resulting violation through penalty terms.

Across the benchmark run, the diagnostic found 398 soft-capacity violation rows. Doorway-capacity overflow occurred mainly in Cells 2, 4, and 6, while safe-threshold overflow occurred mainly in Cells 4, 5, and 7. No physical-capacity overflow occurred in any cell.

Therefore, the model should be interpreted as a soft-constrained CTM. The simulation does not enforce all capacity limits as hard feasibility constraints. Instead, it uses doorway, safe-threshold, and physical-capacity penalties to quantify overload. This is why the physical-capacity penalty remains zero while doorway and safe-threshold penalties are active.

In [88]:
# -capacity diagnostic for on-ramp inflow
# Purpose:c heck whether combined mainline inflow + ramp inflow exceeds doorway capacity, safe threshold, or physical capacity.

print("Soft-Capacity Diagnostic for On-Ramp Inflow")

soft_capacity_rows = []

for t in range(num_steps):
    x_next_step = state_based_benchmark_history["x"][t]
    q_in_step = state_based_benchmark_history["q_in"][t]
    u_in_step = state_based_benchmark_history["u_in"][t]

    for cell in x_next_step:
        total_inflow = q_in_step[cell] + u_in_step[cell]

        doorway_overflow = max(
            total_inflow - doorway_capacity[cell],
            0.0
        )

        safe_overflow = max(
            x_next_step[cell] - safe_threshold_capacity[cell],
            0.0
        )

        physical_overflow = max(
            x_next_step[cell] - physical_capacity[cell],
            0.0
        )

        if (
            doorway_overflow > 1e-9
            or safe_overflow > 1e-9
            or physical_overflow > 1e-9
        ):
            soft_capacity_rows.append({
                "step": t + 1,
                "cell": cell,
                "q_in": q_in_step[cell],
                "u_in": u_in_step[cell],
                "total_inflow": total_inflow,
                "doorway_capacity": doorway_capacity[cell],
                "doorway_overflow": doorway_overflow,
                "x_next": x_next_step[cell],
                "safe_threshold_capacity": safe_threshold_capacity[cell],
                "safe_overflow": safe_overflow,
                "physical_capacity": physical_capacity[cell],
                "physical_overflow": physical_overflow,
            })

soft_capacity_diagnostic_df = pd.DataFrame(soft_capacity_rows)

print("Number of soft-capacity violation rows:", len(soft_capacity_diagnostic_df))

if len(soft_capacity_diagnostic_df) == 0:
    print("PASS: no doorway, safe-threshold, or physical-capacity violations occurred.")
else:
    print("NOTE: capacity violations occurred and were handled through soft penalties.")
    display(soft_capacity_diagnostic_df.round(3))

    summary_df = soft_capacity_diagnostic_df.groupby("cell").agg(
        violation_steps=("step", "count"),
        max_doorway_overflow=("doorway_overflow", "max"),
        max_safe_overflow=("safe_overflow", "max"),
        max_physical_overflow=("physical_overflow", "max"),
    ).reset_index()

    print("Soft-capacity violation summary by cell")
    display(summary_df.round(3))

Soft-Capacity Diagnostic for On-Ramp Inflow
Number of soft-capacity violation rows: 398
NOTE: capacity violations occurred and were handled through soft penalties.


,step,cell,q_in,u_in,total_inflow,doorway_capacity,doorway_overflow,x_next,safe_threshold_capacity,safe_overflow,physical_capacity,physical_overflow
0,1,Cell 2,84.327,8.8,93.127,84.327,8.8,133.000,413.744,0.000,591.062,0.0
1,1,Cell 4,84.327,3.9,88.227,84.327,3.9,101.572,413.744,0.000,591.062,0.0
2,1,Cell 6,84.327,5.6,89.927,84.327,5.6,173.859,413.744,0.000,591.062,0.0
3,2,Cell 2,84.327,8.8,93.127,84.327,8.8,141.800,413.744,0.000,591.062,0.0
4,2,Cell 4,84.327,3.9,88.227,84.327,3.9,105.472,413.744,0.000,591.062,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
393,119,Cell 7,78.327,7.6,85.927,101.192,0.0,630.948,496.492,134.456,709.275,0.0
394,120,Cell 4,70.727,5.4,76.127,84.327,0.0,520.336,413.744,106.592,591.062,0.0
395,120,Cell 5,76.127,8.2,84.327,101.192,0.0,633.148,496.492,136.656,709.275,0.0
396,120,Cell 6,84.327,5.8,90.127,84.327,5.8,341.895,413.744,0.000,591.062,0.0


Soft-capacity violation summary by cell


,cell,violation_steps,max_doorway_overflow,max_safe_overflow,max_physical_overflow
0,Cell 2,23,8.8,0.000,0.0
1,Cell 4,98,5.1,106.992,0.0
2,Cell 5,83,0.0,138.156,0.0
3,Cell 6,120,8.2,0.000,0.0
4,Cell 7,74,0.0,135.756,0.0


In [89]:
# Off-Ramp Outflow Diagnostic
# Purpose:
# Check whether requested off-ramp outflow was ever larger than
# the physically feasible off-ramp outflow used by the CTM update.


off_ramp_diagnostic_rows = []

cell_order = [f"Cell {i}" for i in range(1, 9)]

for t in range(num_steps):
    requested_f_out_step = state_based_benchmark_history["f_out"][t]
    actual_f_out_step = state_based_benchmark_history["actual_f_out"][t]

    for cell in cell_order:
        requested_value = float(requested_f_out_step[cell])
        actual_value = float(actual_f_out_step[cell])

        gap = requested_value - actual_value

        if gap > 1e-9:
            off_ramp_diagnostic_rows.append({
                "step": t + 1,
                "cell": cell,
                "requested_f_out": requested_value,
                "actual_f_out": actual_value,
                "requested_minus_actual": gap,
            })

off_ramp_diagnostic_df = pd.DataFrame(off_ramp_diagnostic_rows)

print("Problem 19: Conservative Off-Ramp Outflow Diagnostic")
print("Total capped off-ramp events:", len(off_ramp_diagnostic_df))

if len(off_ramp_diagnostic_df) > 0:
    print(
        "Max requested-minus-actual off-ramp gap:",
        round(off_ramp_diagnostic_df["requested_minus_actual"].max(), 6)
    )
    display(off_ramp_diagnostic_df.head(20))
    print("NOTE: some requested off-ramp outflows were capped by physical availability.")
else:
    print("Max requested-minus-actual off-ramp gap:", 0.0)
    print("PASS: no off-ramp over-drain occurred in the benchmark run.")

Problem 19: Conservative Off-Ramp Outflow Diagnostic
Total capped off-ramp events: 0
Max requested-minus-actual off-ramp gap: 0.0
PASS: no off-ramp over-drain occurred in the benchmark run.
